# Moteur de recommandation IoT — produits similaires & complementaires

Pour un produit de l'inventaire, ce notebook recommande **les produits similaires** (substituts du meme type) et **les produits complementaires** (a acheter avec).

## Lancer sur Google Colab
1. Mettre `inventory_export.csv` et `regles_recommandation.csv` a cote du notebook (ou les uploader).
2. **Runtime -> Run all** : le modele d'embeddings se telecharge automatiquement (gratuit).
3. Utiliser : `show_smart("ESP32")`, `show_smart("Batterie 18650")`, `evaluer()`.

## Cerveau LLM (optionnel mais recommande)
Un **LLM expert** choisit et **explique** les recommandations parmi les candidats du moteur (sans rien inventer).
Il s'active tout seul si une cle/serveur est present. Trois cas :
- **Societe (PRIVE + ILLIMITE + GRATUIT)** : heberge **Ollama** sur ton serveur (modele `qwen2.5:7b`, licence
  commerciale libre) puis renseigne `LLM_BASE_URL=http://localhost:11434/v1` et `LLM_MODEL=qwen2.5:7b`.
  Le notebook l'utilise EN PRIORITE (`[cerveau: llm:local]`). Aucune donnee ne sort de l'entreprise.
  Voir l'**Annexe** en bas du notebook.
- **Cle API gratuite** : `GROQ_API_KEY` (console.groq.com) ou `GEMINI_API_KEY` (aistudio.google.com) -- plus fluide que HF.
- **HF_TOKEN** (deja la) : fonctionne, mais l'API gratuite HF limite les rafales.

Mets la valeur dans les *Secrets* Colab (cle a gauche) ou dans `.env`. Sans LLM, le moteur fonctionne seul.

## Comment ca marche
- Categorisation + extraction d'attributs (board, chimie, tension, capteur...).
- **SIMILAIRES** (`sim=`) : meme famille + meme attribut definissant ; pour les categories sans definissant
  (mecanique/connectique...), on exige un vrai lien de famille sinon rien. Score = vraie similarite.
- **COMPLEMENTAIRES** (`compat=`) : autre famille, compatibilite dure ; la tension seule ne suffit pas ;
  jamais un outil ni du bruit industriel.
- Embeddings `multilingual-e5-base` ; repli automatique TF-IDF si indisponible.
- Si un LLM est branche, il re-classe et **explique** les recos parmi les candidats du moteur.

## Resultat mesure
Jeu de test etiquete (65 cas reels, matching MOT ENTIER, mesure MOTEUR SEUL) : **precision ~99 %** (similaires ~98 %, complementaires ~100 %), **0 faute grave**. Lance `evaluer()` pour verifier.

## Regler le moteur
Editer `regles_recommandation.csv` (1 ligne/categorie : complement_categories, complement_mots_cles,
exclure_mots_cles, attributs_similaire) puis re-executer a partir de l'etape Regles.


In [ ]:
# === Setup automatique des donnees (Colab) ===
# Recupere inventory_export.csv + regles_recommandation.csv depuis le repo si besoin.
# Sur Colab : clone le repo. En local : ne fait rien si les fichiers sont deja la.
import os, subprocess
if not os.path.exists("inventory_export.csv"):
    if not os.path.isdir("AiRecommadation"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/yazid-mrouki/AiRecommadation.git"])
    if os.path.isdir("AiRecommadation"):
        os.chdir("AiRecommadation")
    print("Donnees pretes dans :", os.path.abspath("."))
else:
    print("Donnees deja presentes dans :", os.path.abspath("."))


In [ ]:
!pip install pandas numpy scikit-learn sentence-transformers rapidfuzz openai anthropic -q
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import re, unicodedata, warnings
warnings.filterwarnings("ignore")
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
    print("OK Dependances -- mode semantique (sentence-transformers)")
except Exception:
    HAS_ST = False
    print("Repli automatique sur TF-IDF (sentence-transformers indisponible)")

## Etape 1 - Chargement des donnees
Recupere `inventory_export.csv` (catalogue Shopify, ~5774 produits) ; sur Colab, clone depuis le repo.

In [ ]:
candidate_files = [
    Path("inventory_export.csv"),
    Path("data/inventory_export.csv"),
    Path("/content/inventory_export.csv"),
    Path("/content/drive/MyDrive/inventory_export.csv"),
]
csv_path = next((p for p in candidate_files if p.exists()), None)
if csv_path is None:
    try:
        from google.colab import files
        print("Selectionne inventory_export.csv a uploader...")
        uploaded = files.upload()
        csv_path = Path(list(uploaded.keys())[0])
    except ModuleNotFoundError:
        raise FileNotFoundError("inventory_export.csv introuvable.")
df_raw = pd.read_csv(csv_path)
print(f"OK Charge depuis : {csv_path}")
print(f"OK {df_raw.shape[0]} lignes, {df_raw['Handle'].nunique()} produits uniques")
df_raw.head(3)

## Etape 2 - Nettoyage & categorisation
Categorie + attributs techniques + **role systeme** (capteur / carte / actionneur / alimentation...) par produit. Objectif **0 % 'autre'**.

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.lower().str.strip()

def normalize_text(text):
    """Texte filtre (sans mots-outils, sans lettres seules) -- pour categories + TF-IDF."""
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii","ignore").decode("ascii").lower()
    text = re.sub(r"[^a-z0-9\s]"," ",text)
    stop = {"de","des","du","la","le","les","un","une","et","au","aux","pour","avec","sur","dans","par","ou","en"}
    return " ".join(t for t in text.split() if len(t)>1 and t not in stop)

def norm_raw(text):
    """Texte brut (GARDE les lettres seules : 'type c', 'pile d') -- pour l'extraction d'attributs."""
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii","ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+"," ",text).strip()

# ============================================================================
# BRIQUE 0 -- CATEGORISATION COMPLETE (objectif : 0 % de "autre")
# ============================================================================
CATEGORY_RULES = {
 "capteur":     ["capteur","capteurs","senseur","sensor","dht11","dht22","ds18b20","hc-sr04","hcsr04","pir","ultrason","mpu6050","mpu9250","bmp","bme280","mq2","mq3","mq135","ldr","acs712","sct013","gy-","lm35","tof","photoresistance","accelerometre","gyroscope","camera","debimetre","yf-s201","flow","ad8232","ecg","hc-sr501","encodeur","mlx","reed","hx711","load cell","fin de course"],
 "carte":       ["raspberry","rpi","arduino","esp32","esp8266","nodemcu","wemos","microbit","stm32","atmega","jetson","wroom","attiny","carte developpement","mega2560","leonardo"],
 "mobilite":    ["voiture","drone","quadcopter","helices","helice","roue","roues","chassis","4wd","2wd","robot","bateau","avion","mecanum","suiveur"],
 "moteur":      ["moteur","motor","servo","servomoteur","stepper","nema17","nema23","28byj","pompe","ventilateur","brushless","pas a pas","mg90","mg90s","mg995","mg996","sg90","ds3218","actionneur","vibreur"],
 "led":         ["led","ruban","bande","ampoule","lampe","lumiere","spectre","rgb","neon","matrice","ws2812","cob","horticole","projecteur","spot","luminaire","strip"],
 "batterie":    ["pile","piles","batterie","battery","18650","21700","26650","14500","lipo","li-po","lithium","accu","alcaline","alkaline","nimh","ni-mh","vape","cr2032","cr2016","cr1620","r20","r14"],
 "chargeur":    ["chargeur","charging","chargement","tp4056","imax"],
 "alimentation":["alimentation","tension","regulateur","regulator","convertisseur","transformateur","decoupage","buck","boost","abaisseur","elevateur","pwm","7805","79l12","onduleur","step-down","step-up","fusible"],
 "rf":          ["433mhz","rf","emetteur","recepteur","bluetooth","ble","wifi","gsm","gprs","gps","nrf24","lora","zigbee","sim800","sim900","antenne","hm-10","nfc","rfid"],
 "electronique":["module","circuit","relais","relay","lcd","oled","hdmi","i2c","spi","74hc","mosfet","registre","potentiometre","potentiometer","ecran","interrupteur","bouton","contacteur","rtc","driver","l298","shield","bouclier","pcb","prototype","perforee","controleur","cnc","gravure","expansion","stc-1000","uln2003","afficheur","segments","tm1637","max7219","clavier","membrane","ft232","rs232","cp2102","ch340","keypad","joystick","dip switch","horloge","ne555"],
 "composant":   ["diode","diodes","resistance","resistances","condensateur","transistor","zener","quartz","cristal","inductance","1n4148","1n4007","thyristor","triac","diac","scr","optocoupleur","triode","optotriac"],
 "connectique": ["cable","cables","fil","fils","connecteur","connecteurs","cosses","cosse","jumper","cavalier","dupont","barette","barrette","prise","borne","bornier","header","nappe","gaine","thermoretractable","domino","carte memoire","micro sd","microsd","sd card","cordon","rallonge","serre cable"],
 "mesure":      ["multimetre","testeur","oscilloscope","amperemetrique","pince ampere","compteur","balance","luxmetre","thermometre","wattmetre","pzem","ph metre","electrode","sonde","frequencemetre","generateur de signal"],
 "soudure":     ["souder","soudure","etain","flux","panne","dessoudage","fer a souder","desoudage"],
 "solaire":     ["solaire","photovoltaique","photovolta","mppt","panneau solaire"],
 "audio":       ["haut-parleur","haut parleur","microphone","amplificateur","buzzer","speaker","ampli","sonore","hp","mp3","jack audio"],
 "outillage":   ["tournevis","cle","cles","clef","pince","pinces","scie","cutter","lame","lames","foret","forets","marteau","coffret","outil","outils","douille","douilles","embout","embouts","percage","brucelle","mandrin","cliquet","perceuse","visseuse","meuleuse","burin","lime","etau","cle mixte","jeu de","pistolet","colle","peinture","disque","brosse","meule","nettoyeur","levage","electrogene","laser","niveau","compresseur","ponceuse","rabot","agrafeuse","riveteuse","decapeur","imprimante","creality","filament","truelle","spatule","pinceau","rouleau","echelle","escabeau","cric","palan","aspirateur","disqueuse","sangle"],
 "mecanique":   ["vis","ecrou","ecrous","boulon","rondelle","rondelles","rivet","roulement","engrenage","courroie","poulie","ressort","rail","profile","entretoise","boitier","coque","dissipateur","heatsink","radiateur","colonnette","equerre","charniere","glissiere","tige filetee"],
 "electrique":  ["disjoncteur","contacteur","sectionneur","chint","parafoudre","goulotte","armoire"],
}
# Mots indiquant un produit dont l'identite PRINCIPALE n'est PAS une carte MCU (meme s'il cite un
# board) : un "Module relais ESP32" est un actionneur, pas une carte -> ne doit pas etre 'carte'.
ACCESSORY = {"lcd","oled","ecran","afficheur","ruban","boitier","coque","dissipateur",
             "ventilateur","camera","shield","transformateur","adaptateur","alimentation","chargeur",
             "relais","relay","detecteur","driver"}
CAT_ORDER = ["chargeur","batterie","led","alimentation","moteur","rf","electronique","composant",
             "connectique","mesure","soudure","solaire","audio","outillage","mecanique","electrique"]

TOOL_BRANDS = {"total","wadfow","harden","yato","tolsen","ingco","stanley","bosch","makita",
               "dewalt","milwaukee","creality","deli"}
def is_tool_brand(title):
    return any(b in norm_raw(title).split() for b in TOOL_BRANDS)

def _has(n, words, kws):
    for k in kws:
        if " " in k:
            if k in n: return True
        elif len(k) <= 3:
            if k in words: return True
        else:
            if any(k in w for w in words) or k in n: return True
    return False

def infer_category(title):
    n = normalize_text(title); words = set(n.split()); wr = set(norm_raw(title).split())
    if _has(n, words, CATEGORY_RULES["mobilite"]): return "mobilite"
    if _has(n, words, CATEGORY_RULES["capteur"]):  return "capteur"
    if _has(n, words, CATEGORY_RULES["carte"]) and not (wr & ACCESSORY): return "carte"
    if _has(n, words, ["relais","relay"]): return "electronique"   # un relais est un ACTIONNEUR, jamais rf/carte (meme s'il est 'WiFi')
    for cat in CAT_ORDER:
        if _has(n, words, CATEGORY_RULES[cat]): return cat
    if is_tool_brand(title): return "outillage"
    return "autre"

# ============================================================================
# SPECS
# ============================================================================
SPEC_PATTERNS = {
 "capacity_mah":(r'\b(\d+(?:[.,]\d+)?)\s*mah\b',50,100000),
 "capacity_ah": (r'\b(\d+(?:[.,]\d+)?)\s*ah\b',0.2,500),
 "voltage_v":   (r'\b(\d+(?:[.,]\d+)?)\s*v\b',0.5,600),
 "power_w":     (r'\b(\d+(?:[.,]\d+)?)\s*w\b',0.1,5000),
 "count":       (r'\b(\d+)\s*(?:pcs|pieces|broches|pin|leds?|canaux|ch)\b',1,10000),
}
def extract_specs(title):
    t = unicodedata.normalize("NFKD", str(title)).encode("ascii","ignore").decode("ascii").lower().replace("*"," ")
    out={}
    for name,(pat,lo,hi) in SPEC_PATTERNS.items():
        v=[float(x.replace(",",".")) for x in re.findall(pat,t)]; v=[x for x in v if lo<=x<=hi]
        if v: out[name]=max(v)
    return out
UPGRADE_SPECS=["capacity_mah","capacity_ah","power_w","count"]
def primary_spec(specs):
    for k in UPGRADE_SPECS:
        if k in specs: return k,specs[k]
    return None,0.0

# ============================================================================
# BRIQUE 1 -- ATTRIBUTS
# ============================================================================
LIION_FF=["18650","21700","26650","14500","16340","18500","20700"]; COIN_FF=["cr2032","cr2016","cr1620","cr2025","cr2450","lr44","ag13"]
def a_form_factor(n):
    for f in LIION_FF:
        if f in n: return f
    for f in COIN_FF:
        if f in n: return "coin"
    if re.search(r'\b(rl20|pile d)\b',n): return "d"
    if re.search(r'\b(rl14|pile c)\b',n): return "c"
    if re.search(r'\b(aaa|lr03|r03)\b',n): return "aaa"
    if re.search(r'\b(aa|lr6|r6)\b',n): return "aa"
    if re.search(r'\b9v\b',n): return "9v"
    if re.search(r'\bnema\s?17\b',n): return "nema17"
    if re.search(r'\bnema\s?23\b',n): return "nema23"
    if "5mm" in n: return "5mm"
    if "3mm" in n: return "3mm"
    for c in ["5050","2835","3528"]:
        if c in n: return c
    return None
def a_chemistry(n,ff):
    if "lipo" in n or "li-po" in n or "polymer" in n: return "lipo"
    if ff in LIION_FF or "li-ion" in n or "liion" in n or "lithium" in n: return "lithium"
    if "nimh" in n or "ni-mh" in n: return "nimh"
    if "alcaline" in n or "alkaline" in n or ff in ("d","c","aa","aaa","9v"): return "alkaline"
    if "plomb" in n or "agm" in n or "ultracell" in n or "acide" in n: return "lead"
    if "vape" in n or "pod" in n: return "vape"
    if ff=="coin": return "lithium"
    return None
def a_connector(n):
    if "type-c" in n or "type c" in n or "usb-c" in n or "usb c" in n: return "usb_c"
    if "micro usb" in n or "micro-usb" in n or "microusb" in n: return "usb_micro"
    if "type-b" in n or "type b" in n or "usb-b" in n: return "usb_b"
    if "xt60" in n: return "xt60"
    if "jst" in n: return "jst"
    if "hdmi" in n: return "hdmi"
    if "rj45" in n or "ethernet" in n: return "rj45"
    if "jack" in n: return "jack"
    if "usb" in n: return "usb_a"
    return None
def a_connectivity(n):
    if "wifi" in n or "wi-fi" in n: return "sans" if ("sans wifi" in n or "without wifi" in n) else "wifi"
    if "bluetooth" in n or re.search(r'\bble\b',n): return "bluetooth"
    if "433" in n or "nrf24" in n or "lora" in n or "zigbee" in n or re.search(r'\brf\b',n): return "rf"
    if "gsm" in n or "gprs" in n: return "gsm"
    if "gps" in n: return "gps"
    if "sans fil" in n: return "wireless"
    return None
def a_control(n):
    if "application" in n or re.search(r'\bappli\b',n) or re.search(r'\bapp\b',n): return "app"
    if "wifi" in n: return "wifi"
    if "bluetooth" in n: return "bluetooth"
    if "telecommand" in n or "radiocommand" in n or re.search(r'\brc\b',n): return "rc"
    return None
def a_vehicle(n):
    w=set(n.split())
    if "voiture" in n or "automobile" in n: return "voiture"
    if "drone" in n or "quadcopter" in n: return "drone"
    if "bateau" in n: return "bateau"
    if "avion" in n: return "avion"
    if "moto" in w: return "moto"
    if "tank" in w: return "char"
    if "robot" in n: return "robot"
    return None
def a_board(n):
    w=set(n.split())
    if "esp32" in n: return "esp32"
    if "esp8266" in n or "nodemcu" in n or "wemos" in n or "esp-12" in n: return "esp8266"
    if "jetson" in n: return "jetson"           # AVANT 'nano' : 'Jetson Nano' n'est PAS un Arduino Nano
    if "raspberry" in n or "rpi" in w or "pi" in w: return "raspberry"
    if "microbit" in n: return "microbit"
    if "stm32" in n: return "stm32"
    if "arduino" in n: return "arduino"
    if w & {"uno","nano","leonardo","mega"}: return "arduino"
    return None
def a_led_form(n):
    if "ruban" in n or "bande" in n or "strip" in n: return "strip"
    # bulb = ampoule a culot (E14/E27/G4/GU10/bougie/capsul). PAS un projecteur ni une lampe frontale.
    if ("ampoule" in n or re.search(r'\be14\b',n) or re.search(r'\be27\b',n) or "gu10" in n
            or re.search(r'\bg4\b',n) or "bougie" in n or "capsul" in n): return "bulb"
    if "matrice" in n or "ws2812" in n or "8x8" in n or "neopixel" in n: return "matrix"
    if "cob" in n or "horticole" in n or "spectre" in n: return "cob"
    if "infrarouge" in n or "850nm" in n or "940nm" in n or re.search(r'\bir\b',n): return "ir"
    if "afficheur" in n or "7 segment" in n or "segments" in n: return "display"
    if "neon" in n: return "neon"
    if "5mm" in n or "3mm" in n: return "discrete"
    # projecteur / spot / lampe frontale / torche = luminaire, PAS une ampoule a culot
    if "projecteur" in n or "spot" in n or "luminaire" in n or "frontale" in n or "torche" in n or "lampadaire" in n: return "projector"
    if "lampe" in n: return "bulb"
    return None
def a_voltage_bucket(specs):
    v=specs.get("voltage_v")
    if v is None: return None
    for b in (3.3,5,12,24,220):
        if abs(v-b)<=max(0.6,b*0.12): return b
    return round(v)
def a_voltage_domain(specs):
    # DOMAINE de tension (regle dure de compatibilite) :
    #   logic = 3.3/5 V (MCU, capteurs) | lv = 12/24/48 V (rubans, moteurs, alim DC) | mains = 110/220/380 V (secteur)
    v=specs.get("voltage_v")
    if v is None: return None
    if v<=6: return "logic"
    if v<=60: return "lv"
    return "mains"
def a_psu(n):
    if "buck" in n or "abaisseur" in n or "step down" in n: return "buck"
    if "boost" in n or "elevateur" in n or "step up" in n: return "boost"
    if "decoupage" in n or "transformateur" in n or "ac dc" in n: return "acdc"
    if "pwm" in n: return "pwm"
    return None
def a_rftech(n):
    if "433" in n: return "433"
    if "bluetooth" in n or re.search(r'\bble\b',n) or "hm-10" in n: return "bluetooth"
    if "wifi" in n: return "wifi"
    if "gsm" in n or "gprs" in n or "sim800" in n: return "gsm"
    if "gps" in n: return "gps"
    if "lora" in n: return "lora"
    if "zigbee" in n: return "zigbee"
    if "nrf24" in n: return "nrf24"
    if "nfc" in n or "rfid" in n: return "nfc"
    return None
def a_component(n):
    # diac/scr AVANT diode (sous-types distincts : un DIAC declencheur != une diode, != un TRIAC).
    for c in ["resistance","condensateur","transistor","zener","diac","scr","diode","quartz","inductance","optocoupleur","thyristor","triac"]:
        if c in n: return c
    return None
def a_module_fn(n):
    # reconnait la FONCTION d'un circuit / module (registre, timer, rtc, driver, ampli...)
    if "relais" in n or "relay" in n: return "relais"
    if "registre" in n or "decalage" in n or "shift register" in n or "74hc595" in n or "74hc164" in n or "74hc165" in n: return "registre"
    if "rtc" in n or "horloge" in n or "ds1302" in n or "ds3231" in n: return "rtc"
    if "ne555" in n or "timer" in n or re.search(r'\b555\b',n): return "timer"
    # display = un AFFICHEUR, PAS un objet qui contient un ecran (stylo 3D a ecran, jauge d'angle
    # LCD, fer a souder a ecran, oscilloscope...). Sinon ces objets deviennent "substituts" d'un LCD.
    if ("lcd" in n or "oled" in n or "afficheur" in n or "tft" in n or "nextion" in n
            or "1602" in n or "2004" in n or "12864" in n or "7 segment" in n or "segments" in n
            or re.search(r'\becran\b',n)):
        if not re.search(r'\b(stylo|jauge|fer a souder|imprimante|niveau|tournevis|3d|socle|angle|'
                         r'oscilloscope|microscope|loupe|camera|televiseur|moniteur|balance|'
                         r'thermometre|multimetre|testeur|pince|perceuse|visseuse)\b', n):
            return "display"
    if "l298" in n or "uln2003" in n or "driver" in n or "darlington" in n: return "driver"
    if "amplificateur" in n or "opamp" in n or "lm358" in n or "lm741" in n or "tda" in n: return "ampli"
    if "potentiometre" in n or "potentiometer" in n: return "potentiometre"
    if "optocoupleur" in n: return "opto"
    if "convertisseur" in n: return "convertisseur"
    if "interrupteur" in n: return "interrupteur"
    if "bouton" in n: return "bouton"
    if re.search(r'\b74(hc|hct|ls)\d',n): return "logique"
    return None
def a_motor(n):
    if "servo" in n or re.search(r'\b(mg90|mg90s|mg995|mg996|sg90|ds3218)\b',n): return "servo"
    if "stepper" in n or "nema" in n or "28byj" in n or "pas a pas" in n: return "stepper"
    if "pompe" in n: return "pompe"
    if "ventilateur" in n: return "ventilateur"
    if "moteur" in n or "motor" in n: return "dc"
    return None
def a_sensor(n):
    if "camera" in n: return "camera"
    # sous-types SPECIFIQUES d'abord (un capteur de gestes/debit n'est pas un capteur de mouvement) :
    if "geste" in n or "gestuelle" in n or "apds" in n or "paj76" in n or "gy-99" in n or "gy 99" in n: return "geste"
    if "debit" in n or "debitmetre" in n or "yf-s" in n or "yf s" in n or "57qf" in n: return "debit"
    if "niveau d eau" in n or "niveau eau" in n or "flotteur" in n: return "niveau"
    # TEMPERATURE -- sous-types par FONCTION EXACTE + INTERFACE (substituabilite STRICTE) :
    #   un DHT (temp+humidite, numerique) n'est PAS un substitut d'un DS18B20 (temp seule 1-wire),
    #   ni d'un LM35 (analogique), ni d'un PT100 (RTD industriel). Chaque interface = sa famille.
    if ("dht" in n or "am2302" in n or "am2301" in n or "sht" in n or "aht" in n or "htu" in n
            or "si70" in n or "hdc1080" in n
            or ("temperature" in n and ("humidite" in n or "humidity" in n))): return "temp_humid"   # temp+humidite numerique
    if "pt100" in n or "pt1000" in n or "thermocouple" in n or "max6675" in n or "max31855" in n or "type k" in n: return "temp_rtd"  # RTD/TC industriel analogique
    if "ds18b20" in n or "ds18s20" in n or "max31820" in n: return "temp_1wire"          # temperature seule numerique 1-wire
    if "lm35" in n or "tmp36" in n or re.search(r'ntc', n): return "temp_analog"     # capteur IC analogique
    if "temperature" in n or "thermom" in n: return "temp_generic"                       # temperature, techno non precisee
    if "hc-sr04" in n or "ultrason" in n or "sharp" in n or "tof" in n or "distance" in n: return "distance"
    if re.search(r'\bpir\b',n) or "mouvement" in n or "motion" in n or "hc-sr501" in n: return "motion"
    if re.search(r'\bmq\b',n) or "mq-" in n or "gaz" in n or "co2" in n or "pollution" in n: return "gaz"
    if "bmp" in n or "bme" in n or "pression" in n: return "pression"
    if "acs712" in n or "sct013" in n or "courant" in n: return "courant"
    # NB : \bmpu\b (et non "mpu") sinon "impulsionnelle" matcherait MPU6050 par sous-chaine.
    if re.search(r'\b(mpu6050|mpu9250|mpu|gy-521|gyroscope|accelerometre)\b',n): return "imu"
    if "ldr" in n or "lux" in n or "photoresistance" in n: return "lumiere"
    if "humidite" in n or "pluie" in n: return "humidite"
    return None

def extract_attrs(title, specs):
    n = norm_raw(title); ff = a_form_factor(n)
    return {"form_factor":ff,"chemistry":a_chemistry(n,ff),"connector":a_connector(n),
            "connectivity":a_connectivity(n),"control":a_control(n),"vehicle":a_vehicle(n),
            "board":a_board(n),"led_form":a_led_form(n),"voltage_bucket":a_voltage_bucket(specs),
            "voltage_domain":a_voltage_domain(specs),
            "psu":a_psu(n),"rftech":a_rftech(n),"component":a_component(n),
            "module_fn":a_module_fn(n),"motor":a_motor(n),"sensor":a_sensor(n)}

# ============================================================================
# ETAPE 2 du raisonnement : TYPE / POSITION dans la CHAINE SYSTEME IoT
#   alimentation -> controle(MCU) -> capteur -> traitement -> actionneur -> interface
#   + communication (RF, rattachee au controle) ; mecanique/connectique = supports ;
#   + outil = HORS systeme IoT. C'est ce ROLE (pas le texte) qui autorise/refuse une relation.
# ============================================================================
_CAT_ROLE = {
 "batterie":"alimentation","chargeur":"alimentation","alimentation":"alimentation","solaire":"alimentation",
 "carte":"controle",
 "capteur":"capteur","mesure":"capteur",
 "rf":"communication",
 "moteur":"actionneur","mobilite":"actionneur",
 "led":"interface","audio":"interface",
 "composant":"traitement",
 "connectique":"connectique","mecanique":"mecanique",
 "outillage":"outil","electrique":"outil",
}
def a_system_position(category, attrs):
    """Role systeme d'un produit. 'electronique' est un fourre-tout -> affine par module_fn
    (+ presence d'un capteur : un thermostat 'contient' un capteur -> role=capteur, donc PAS un
    complement d'un ruban LED car une interface ne se branche pas sur un capteur)."""
    a = attrs or {}
    if category == "electronique":
        mf = a.get("module_fn")
        if mf == "display":                                         return "interface"
        if mf in ("relais","driver"):                               return "actionneur"
        if mf == "convertisseur":                                   return "alimentation"
        if mf in ("ampli","bouton","interrupteur","potentiometre"): return "interface"
        if a.get("sensor"):                                         return "capteur"
        return "traitement"
    return _CAT_ROLE.get(category, "autre")

# ============================================================================
# Construction des colonnes
# ============================================================================
df["product_handle"] = df["handle"].astype(str).str.strip()
df["product_title"]  = df["title"].astype(str).str.strip()
df["sku"]            = df["sku"].astype(str).str.strip() if "sku" in df.columns else ""

for col in ["available (not editable)","on hand (current)","on hand (new)"]:
    if col in df.columns:
        df[col] = df[col].replace("not stocked",0)
        df[col] = pd.to_numeric(df[col],errors="coerce").fillna(0).astype(int)

df["clean_title"] = df["product_title"].apply(normalize_text)
df["category"]    = df["product_title"].apply(infer_category)

# Repli INTELLIGENT : tout produit encore "autre" herite de la categorie de son plus proche voisin
_mask = df["category"] == "autre"
if _mask.any():
    _known = df.loc[~_mask]
    _v  = TfidfVectorizer(max_features=3000).fit(df["clean_title"])
    _nn = NearestNeighbors(n_neighbors=1, metric="cosine").fit(_v.transform(_known["clean_title"]))
    _, _ind = _nn.kneighbors(_v.transform(df.loc[_mask, "clean_title"]))
    df.loc[_mask, "category"] = _known["category"].to_numpy()[_ind[:, 0]]

df["specs"]   = df["product_title"].apply(extract_specs)
df["attrs"]   = df.apply(lambda r: extract_attrs(r["product_title"], r["specs"]), axis=1)
df["is_tool"] = df["product_title"].apply(is_tool_brand)

uniq = df.drop_duplicates("product_handle")
print("OK Categorisation complete + attributs extraits")
dist = uniq["category"].value_counts()
print("\nDistribution des categories :")
print(dist.to_string())
print(f"\n   'autre' = {100*dist.get('autre',0)/len(uniq):.1f}%   "
      f"| produits avec >=1 attribut = {100*(uniq['attrs'].apply(lambda a: any(v is not None for v in a.values()))).mean():.0f}%")

In [ ]:
# === Affinage des categories (matching sur titre NORMALISE : sans accents/ponctuation) ===
_tn = df["product_title"].map(norm_raw)

# ============================================================================
# 0) FAUTE "POUR <board>" : un ACCESSOIRE pour Arduino/ESP/RPi mal range en 'carte'
#    (ex: "jeux de fils ... pour arduino") -> remis dans sa VRAIE famille.
#    On ne touche PAS aux vraies cartes (uno/nano/mega/wroom/dev kit) ni au "+ cable" final.
# ============================================================================
_pour = _tn.str.contains(r"pour (arduino|esp32|esp8266|raspberry|rpi|micro ?bit|stm32|nodemcu)|compatible (arduino|raspberry)", na=False)
_board_noun = _tn.str.contains(r"\buno\b|\bnano\b|\bmega\b|wroom|wrover|dev.?kit|carte de developpement|leonardo|\bpico\b|atmega|esp32 s3|esp32 c3|esp 12|jetson", na=False)
_acc = (df["category"]=="carte") & _pour & (~_board_noun)
df.loc[_acc, "category"] = "connectique"                                                                      # defaut : cables / fils
df.loc[(df["category"]=="connectique") & _pour & _tn.str.contains(r"\becran\b|afficheur|\blcd\b|oled|matrice|shield|bouclier|module", na=False), "category"] = "electronique"
df.loc[(df["category"]=="connectique") & _pour & _tn.str.contains(r"boitier|coque|housse|dissipateur|\bsupport\b|ventilateur|radiateur", na=False), "category"] = "mecanique"
df.loc[(df["category"]=="connectique") & _pour & _tn.str.contains(r"\bchargeur\b|\balimentation\b|adaptateur secteur|transformateur", na=False), "category"] = "alimentation"
# retirer le faux board de ces accessoires (sinon ils matchent les cartes a tort)
for _i in df.index[_acc]:
    _a = dict(df.at[_i, "attrs"]); _a["board"] = None; df.at[_i, "attrs"] = _a

# 1) AUDIO (enceintes, transmetteur FM, recepteur audio) -> 'audio'
_audio = _tn.str.contains(r"haut parleur|enceinte|speaker|soundtech|barre de son|"
                          r"transmetteur fm|transmetteur audio|adaptateur audio|recepteur audio|mains libre", na=False)
df.loc[_audio, "category"] = "audio"

# 2) INDUSTRIEL (automates, PLC, disjoncteurs...) -> 'electrique' (aucun complement IoT)
_indus = _tn.str.contains(r"automate|simatic|s7 1200|s7 200|s7 300|\bplc\b|industrial|industruino|"
                          r"disjoncteur|contacteur|sectionneur|variateur|profibus|\bvfd\b|triphas", na=False)
df.loc[_indus, "category"] = "electrique"

# 3) OUTILS (moteur auto + electroportatif) dans 'moteur' -> 'outillage'
_tool = _tn.str.contains(r"compressiometre|compression moteur|soupape|depose|calage|distribution|vilebrequin|"
                         r"culasse|injecteur|support moteur|cale moteur|extracteur|arrache|demonte|"
                         r"cle a choc|visseuse|perceuse|meuleuse|disqueuse|cisaille|souffleur|ponceuse|rabot|"
                         r"\bscie\b|testeur.*moteur", na=False)
df.loc[_tool & (df["category"]=="moteur"), "category"] = "outillage"

# 4) Cartes de developpement (camera incluse) -> 'carte'
_devboard = (_tn.str.contains(r"carte de developpement|carte developpement|dev board|dev kit|wroom|wrover|nodemcu|esp32 cam|esp cam", na=False)
             & _tn.str.contains(r"esp32|esp8266|esp 12|arduino|raspberry|stm32|rp2040|pico|microbit", na=False))
df.loc[_devboard, "category"] = "carte"

# 4b) Une carte ESP32/ESP8266 SANS fonction de module = une CARTE
_espcard = df["attrs"].apply(lambda a: a.get("board") in ("esp32","esp8266","rp2040") and a.get("module_fn") is None) \
           & df["category"].isin(["capteur","rf","electronique"])
df.loc[_espcard, "category"] = "carte"

# 5) Accessoires de BATTERIE -> vraie categorie
df.loc[_tn.str.contains(r"\btesteur\b", na=False) & (df["category"]=="batterie"), "category"] = "mesure"
df.loc[_tn.str.contains(r"\bbms\b|carte de protection|protection batterie", na=False) & (df["category"]=="batterie"), "category"] = "composant"
df.loc[_tn.str.contains(r"boitier|power bank|porte pile", na=False) & (df["category"]=="batterie"), "category"] = "mecanique"

# 6) FPGA / Altera ne sont PAS des Arduino -> corriger le faux attribut board
_fpga = _tn.str.contains(r"fpga|altera|cyclone|spartan|xilinx|lattice", na=False)
for _i in df.index[_fpga]:
    _a = dict(df.at[_i, "attrs"]); _a["board"] = "fpga"; df.at[_i, "attrs"] = _a

# 7) Panneau solaire mal classe : le mot "panneau" CONTIENT "panne" (mot-cle soudure) -> faux match.
_solpan = _tn.str.contains(r"panneau solaire|panneau photovolta|cellule solaire|mono ?cristallin|polycristallin|module solaire", na=False)
df.loc[_solpan, "category"] = "solaire"

# 8) Condensateurs CBB61/CBB65 (ventilateur/demarrage/climatiseur) mal classes 'moteur' -> composant (secteur 220/450V).
_cbb = _tn.str.contains(r"\bcbb6\d\b|condensateur.*(ventilateur|demarrage|moteur|climatiseur)", na=False) & (df["category"]=="moteur")
df.loc[_cbb, "category"] = "composant"

# 9) Faux 'moteur' : rapporteur d'angle (mesure), pompe nettoyant/tuyau/boite vide (outillage/mecanique).
df.loc[_tn.str.contains(r"rapporteur", na=False) & (df["category"]=="moteur"), "category"] = "mesure"
df.loc[_tn.str.contains(r"nettoyant|tuyau|boite vide|separation d ecran", na=False) & (df["category"]=="moteur"), "category"] = "outillage"

# 10) Consommables auto/bricolage mal classes 'moteur' (le mot "pompe" -> a_motor=pompe) :
#     pompe a retouche/peinture, WD-40, degrippant, mastic, vernis, degraissant... -> outillage.
_consool = _tn.str.contains(r"pompe a retouche|pompe retouche|retouche|wd[- ]?40|degrippant|degraissant|"
                            r"mastic|vernis|silicone|graisse|lubrifiant|peinture|scaner|scanner", na=False) & (df["category"]=="moteur")
df.loc[_consool, "category"] = "outillage"

# 11) Adaptateurs/convertisseurs VIDEO (HDMI/VGA/RCA/DVI/AV/peritel) ne sont PAS de l'alimentation :
#     ce sont de la connectique video. (Evite "adaptateur HDMI-VGA" propose pour un ruban LED / une alim.)
_video = _tn.str.contains(r"hdmi|\bvga\b|\brca\b|\bdvi\b|peritel|displayport|av audio|video", na=False) \
         & df["category"].isin(["alimentation","electronique"]) \
         & ~_tn.str.contains(r"alimentation|regulateur|convertisseur (ac|dc|cc)|multitension|chargeur", na=False)
df.loc[_video, "category"] = "connectique"

# 12) SCR / DIAC / TRIAC / thyristor sont des SEMI-CONDUCTEURS -> composant (pas mecanique).
df.loc[_tn.str.contains(r"\bscr\b|\bdiac\b|\btriac\b|thyristor", na=False) & (df["category"]=="mecanique"), "category"] = "composant"

# 13) Kits/testeurs AUTO (pression frein/huile/culasse/injection/essence, purge, etrier...) -> outillage.
_autotool = _tn.str.contains(r"pression (frein|huile|injection|essence|carburant)|etrier|culasse|"
                             r"purge|liquide de frein|compression|manometre|separateur|sonde lambda", na=False) & (df["category"]=="mesure")
df.loc[_autotool, "category"] = "outillage"

_nb_acc = int(_acc.sum())
print(f"OK Affinage : {_nb_acc} accessoires 'pour <board>' reclasses + audio/industriel/outils/ESP/batterie/FPGA + solaire/CBB")

## Etape 3 - Regles de recommandation
Matrice de complementarite **editable** (`regles_recommandation.csv`).

In [ ]:
# ============================================================================
# REGLES DE RECOMMANDATION (source = ce notebook ; exportees en CSV editable)
#   colonnes : attributs_similaire | complement_categories |
#              complement_categories_toutes | complement_mots_cles |
#              exclure_mots_cles  (NOUVEAU : blackliste de complements)
# IMPORTANT : REGEN_RULES=True reecrit toujours le CSV depuis _DEFAULT_RULES.
#   -> Pour personnaliser : edite _DEFAULT_RULES ci-dessous, puis re-execute a partir de l'Etape 5.
# ============================================================================
REGEN_RULES = True
RULES_CSV = Path("regles_recommandation.csv")

_DEFAULT_RULES = [
 {"categorie":"batterie","attributs_similaire":"form_factor;chemistry","complement_categories":"chargeur;composant;mecanique;connectique","complement_categories_toutes":"chargeur","complement_mots_cles":"bms;protection;support;porte pile;boitier;holder;chargeur;18650;module charge","exclure_mots_cles":"diode;triac;mosfet;transistor;resistance;condensateur;foret;projecteur;capteur;relais;moteur;ruban led","exemple":"18650 -> chargeur 18650 / BMS / support (jamais diode/TRIAC)"},
 {"categorie":"chargeur","attributs_similaire":"chemistry;connector","complement_categories":"batterie;connectique","complement_categories_toutes":"batterie","complement_mots_cles":"batterie;pile;accu;18650;lipo;li-ion;cellule","exclure_mots_cles":"foret;vis;peinture;moteur","exemple":"Chargeur 18650 -> cellules 18650"},
 {"categorie":"led","attributs_similaire":"led_form;voltage_bucket","complement_categories":"alimentation;electronique;connectique","complement_categories_toutes":"","complement_mots_cles":"alimentation;transformateur;controleur;dimmer;connecteur ruban;profile;amplificateur rgb;bande;ruban","exclure_mots_cles":"foret;vis;peinture;moteur;capteur;fpga;altera;cyclone;diode;projecteur","exemple":"Ruban LED 12V -> alim 12V / controleur RGB 12V"},
 {"categorie":"alimentation","attributs_similaire":"voltage_bucket;connector","complement_categories":"carte;connectique;led;electronique;mecanique","complement_categories_toutes":"","complement_mots_cles":"raspberry;arduino;esp32;micro sd;carte memoire;boitier;dissipateur;ruban led;ruban;jack","exclure_mots_cles":"foret;vis;peinture;cle;douille;fpga;altera;cyclone;diode;triac;projecteur;domino","exemple":"Alim 5V Raspberry -> Raspberry / carte SD / boitier / dissipateur"},
 {"categorie":"carte","attributs_similaire":"board;connectivity","complement_categories":"capteur;connectique;alimentation;led;electronique;composant","complement_categories_toutes":"capteur","complement_mots_cles":"breadboard;plaque essai;dupont;jumper;cable usb;micro sd;carte memoire;ecran;oled;lcd;afficheur;capteur;alimentation;boitier;dissipateur;relais","exclure_mots_cles":"foret;perceuse;vis;cle;douille;peinture;disjoncteur;fpga;altera;cyclone;visseuse;clé à chocs","exemple":"Arduino/ESP -> capteurs / dupont / alim / ecran"},
 {"categorie":"capteur","attributs_similaire":"sensor","complement_categories":"carte;connectique;electronique","complement_categories_toutes":"carte","complement_mots_cles":"arduino;esp32;raspberry;dupont;jumper;breadboard;resistance;ecran;lcd;oled;module;relais","exclure_mots_cles":"foret;perceuse;vis;cle;peinture;fpga;altera;cyclone;clé à chocs;projecteur","exemple":"Capteur DHT11/PIR -> carte MCU / dupont / ecran"},
 {"categorie":"moteur","attributs_similaire":"motor;voltage_bucket","complement_categories":"carte;electronique;alimentation;batterie;mecanique","complement_categories_toutes":"","complement_mots_cles":"driver;l298;a4988;controleur;esc;arduino;esp32;roue;helice;batterie;lipo;alimentation;servo;pont en h","exclure_mots_cles":"foret;perceuse;vis;peinture;clé à chocs;cle a choc;visseuse;disqueuse;meuleuse;fpga","exemple":"Moteur -> driver L298/ESC / carte / batterie"},
 {"categorie":"mobilite","attributs_similaire":"vehicle;control","complement_categories":"batterie;moteur;capteur;electronique;connectique;chargeur;rf","complement_categories_toutes":"moteur","complement_mots_cles":"telecommande;radiocommande;roue;helice;chassis;lipo;li-ion;18650;nimh;driver;l298;controleur;esc;ultrason;hc-sr04;suiveur;dupont;moteur;servo;recepteur","exclure_mots_cles":"foret;vis;peinture;cle;douille;perceuse;enceinte;haut parleur;lora;audio;jack;fpga;nema;cnc;industriel;graisse;gonfleur","exemple":"Voiture RC -> moteur / driver / batterie / capteur ultrason"},
 {"categorie":"rf","attributs_similaire":"rftech","complement_categories":"carte;alimentation;connectique","complement_categories_toutes":"","complement_mots_cles":"arduino;esp32;raspberry;antenne;dupont;module;carte sim;adaptateur","exclure_mots_cles":"foret;vis;haut parleur;enceinte;fpga;altera;peinture","exemple":"NRF24/LoRa/433 -> carte MCU / antenne / dupont"},
 {"categorie":"electronique","attributs_similaire":"module_fn","complement_categories":"carte;connectique;composant;alimentation","complement_categories_toutes":"","complement_mots_cles":"arduino;raspberry;esp32;nano;uno;dupont;breadboard;resistance;potentiometre;support;afficheur;ecran;lcd;oled;capteur","exclure_mots_cles":"foret;perceuse;vis;cle;douille;peinture;fpga;altera;cyclone;projecteur;clé à chocs","exemple":"Relais/LCD/Module -> Arduino / dupont / resistance / afficheur"},
 {"categorie":"composant","attributs_similaire":"component","complement_categories":"electronique;connectique;carte","complement_categories_toutes":"","complement_mots_cles":"breadboard;plaque essai;dupont;resistance;support;arduino;esp32;led;afficheur;potentiometre","exclure_mots_cles":"foret;perceuse;vis;peinture;fpga;altera;cyclone;projecteur;clé à chocs","exemple":"Resistance/diode -> breadboard / dupont / support DIP"},
 {"categorie":"connectique","attributs_similaire":"connector","complement_categories":"carte;electronique;composant","complement_categories_toutes":"","complement_mots_cles":"arduino;raspberry;esp32;module;breadboard;capteur;ecran;led;afficheur","exclure_mots_cles":"foret;vis;peinture;cle;douille;fpga;altera","exemple":"Cable dupont -> carte / module / breadboard"},
 {"categorie":"mesure","attributs_similaire":"","complement_categories":"composant;connectique","complement_categories_toutes":"","complement_mots_cles":"sonde;electrode;pince;cable;cordon;fusible;pile 9v;crocodile","exclure_mots_cles":"foret;vis;peinture;arduino;esp32;fpga","exemple":"Multimetre -> sondes / cordons / pinces"},
 {"categorie":"soudure","attributs_similaire":"","complement_categories":"soudure;composant;connectique","complement_categories_toutes":"","complement_mots_cles":"etain;flux;panne;tresse;pompe a dessouder;troisieme main;decapant","exclure_mots_cles":"foret;perceuse;vis;arduino;esp32;fpga;fer a souder;poste soudure;masque","exemple":"Fer a souder -> etain / flux / panne (consommables)"},
 {"categorie":"solaire","attributs_similaire":"voltage_bucket","complement_categories":"batterie;alimentation;chargeur","complement_categories_toutes":"","complement_mots_cles":"batterie;controleur;mppt;regulateur;18650;lipo;onduleur","exclure_mots_cles":"foret;vis;peinture;fpga;arduino","exemple":"Panneau solaire -> regulateur MPPT / batterie"},
 {"categorie":"audio","attributs_similaire":"","complement_categories":"connectique;alimentation;chargeur","complement_categories_toutes":"","complement_mots_cles":"jack;aux;cable;chargeur;alimentation;support;amplificateur;haut parleur;micro;rca","exclure_mots_cles":"foret;vis;esp32;arduino;relais;fpga;capteur;ruban","exemple":"Enceinte BT -> cable jack / ampli / alim"},
 {"categorie":"outillage","attributs_similaire":"","complement_categories":"","complement_categories_toutes":"","complement_mots_cles":"","exclure_mots_cles":"","exemple":"Outil -> aucun complement IoT"},
 {"categorie":"mecanique","attributs_similaire":"","complement_categories":"","complement_categories_toutes":"","complement_mots_cles":"","exclure_mots_cles":"","exemple":"Boitier/vis -> aucun complement direct (sert aux cartes)"},
 {"categorie":"electrique","attributs_similaire":"","complement_categories":"","complement_categories_toutes":"","complement_mots_cles":"","exclure_mots_cles":"","exemple":"Disjoncteur/220V -> aucun complement IoT"},
]
if REGEN_RULES or not RULES_CSV.exists():
    pd.DataFrame(_DEFAULT_RULES).to_csv(RULES_CSV, index=False)
    print(f"Regles ecrites dans {RULES_CSV}  (REGEN_RULES={REGEN_RULES})")

_rules = pd.read_csv(RULES_CSV).fillna("")
def _spl(s): return [x.strip() for x in str(s).split(";") if x.strip()]

TYPE_ATTRIBUTES     = {r["categorie"]: _spl(r["attributs_similaire"])          for _,r in _rules.iterrows() if _spl(r["attributs_similaire"])}
COMPLEMENTARITY_MAP = {r["categorie"]: _spl(r["complement_categories"])        for _,r in _rules.iterrows() if _spl(r["complement_categories"])}
COMPLEMENT_BROAD    = {r["categorie"]: _spl(r["complement_categories_toutes"]) for _,r in _rules.iterrows()}
COMPLEMENT_KEYWORDS = {r["categorie"]: [normalize_text(k) for k in _spl(r["complement_mots_cles"])] for _,r in _rules.iterrows()}
COMPLEMENT_EXCLUDE  = {r["categorie"]: [normalize_text(k) for k in _spl(r.get("exclure_mots_cles",""))] for _,r in _rules.iterrows()}

print(f"OK Regles chargees -- {len(_rules)} categories (avec blackliste)")
_rules[["categorie","complement_mots_cles","exclure_mots_cles"]].head(6)


## Etape 4 - Agregation des produits
Produits uniques + stock + controles qualite (sanity checks).

In [ ]:
# ETAPE 2 (TYPE SYSTEME) : role dans la chaine, calcule sur la categorie FINALE (post-affinage).
df["attrs"] = [{**a, "system_position": a_system_position(c, a)} for a, c in zip(df["attrs"], df["category"])]
products = df.drop_duplicates(subset=["product_handle"]).copy()
inv = df.groupby("product_handle").agg({"available (not editable)":"sum","on hand (current)":"sum","on hand (new)":"sum"}).reset_index()
inv.columns = ["product_handle","available_total","on_hand_current","on_hand_new"]
products = products.merge(inv, on="product_handle", how="left")
products["available_total"] = products["available_total"].fillna(0).astype(int)
products["product_profile"] = products["product_title"].fillna("") + " categorie " + products["category"].fillna("")
products = products[["product_handle","product_title","category","sku","specs","attrs","is_tool","clean_title","product_profile","available_total","on_hand_current","on_hand_new"]].reset_index(drop=True)
print(f"OK {len(products)} produits uniques | En stock: {len(products[products['available_total']>0])}")
products[["product_title","category","attrs","available_total"]].head(5)

# --- SANITY CHECKS (qualite pro) : echoue tot et clairement si les donnees sont anormales ---
assert len(products) > 1000, "Catalogue trop petit : les 2 CSV sont-ils bien charges ?"
assert (products["category"]=="autre").mean()==0, "Des produits restent en categorie 'autre'"
_vc = products["category"].value_counts()
assert (_vc>0).all() and len(_vc)>=15, "Une categorie est vide ou il en manque"
print(f"OK sanity : {len(products)} produits, {len(_vc)} categories, 0% 'autre'")


## Etape 5 - Cles LLM (optionnel)
Lues depuis les **Secrets Colab** (`GROQ_API_KEY`, `GEMINI_API_KEY`...). Sans cle -> mode **moteur seul** (deterministe, 99 %). Rotation auto si plusieurs cles Groq.

In [ ]:
import os
def _load_dotenv(path=".env"):
    p = Path(path)
    if not p.exists(): return
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
_KEYS = ["LLM_BASE_URL","LLM_MODEL","HF_TOKEN","GEMINI_API_KEY","GROQ_API_KEY","ANTHROPIC_API_KEY","OPENAI_API_KEY"]
# Cles Groq de SECOURS (rotation auto quand une sature) : nomme-les GROQ_API_KEY_2, _3... dans les Secrets.
_GROQ_BACKUP = ["GROQ_API_KEY_%d" % n for n in range(2, 6)]
try:
    from google.colab import userdata
    for _k in _KEYS + _GROQ_BACKUP:
        try:
            _v = userdata.get(_k)
            if _v: os.environ[_k] = _v
        except Exception:
            pass
except Exception:
    _load_dotenv()
if os.getenv("HF_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = os.getenv("HF_TOKEN")
_found = [k for k in _KEYS if os.getenv(k)]
_ngroq = sum(1 for k in (["GROQ_API_KEY"] + _GROQ_BACKUP) if os.getenv(k))
print("Cles LLM detectees :", _found if _found else "aucune (le moteur fonctionne sans LLM)")
if _ngroq > 1: print(f"  -> {_ngroq} cles Groq chargees : rotation auto quand une atteint sa limite")


## Etape 6 - Moteur de recommandation (deterministe)
**Similaires** = substitut technique (meme role + meme attribut definissant) ; **Complements** = chaine systeme (roles adjacents). Les embeddings recuperent les candidats, les **regles decident**.

In [ ]:
# ============================================================================
# Etape 5 -- MOTEUR DE RECOMMANDATION v2 (semantique E5) + REGLES DURES (deterministe)
#   SIMILAIRES (sim=)    : meme famille + meme attribut DEFINISSANT (pas de cross-architecture).
#   COMPLEMENTAIRES (compat=) : meme chaine systeme (<=2 sauts) ; compatibilite dure ; regle de
#                          TENSION (logic/lv/mains) ; statut compat_status (compatible / necessite
#                          adaptation) ; jamais un outil. Repli TF-IDF si E5 indisponible.
# ============================================================================
try:
    from rapidfuzz import process as _rf_process, fuzz as _rf_fuzz
    HAS_FUZZ = True
except Exception:
    HAS_FUZZ = False

# ---------------------------------------------------------------- engine
class SmartRecoEngine:
    EMB_MODEL    = "intfloat/multilingual-e5-base"
    RERANK_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
    STRONG = re.compile(r'^(?=.*[a-z])(?=.*\d)[a-z0-9]{3,}$')
    BLACKLIST = {"220v","380v","secteur","automate","simatic","plc","disjoncteur","contacteur",
                 "sectionneur","industrial","triphase","cn3791","variateur","profibus","fpga","altera","cyclone","cbb65","climatiseur"}
    # Attribut DEFINISSANT : doit coincider pour qu'un produit soit un VRAI substitut.
    DEFINING = {"batterie":"form_factor","carte":"board","composant":"component","capteur":"sensor",
                "electronique":"module_fn","moteur":"motor","led":"led_form","rf":"rftech","mobilite":"vehicle"}
    # "REASONING FILTER" : un COMPLEMENT doit etre un maillon ADJACENT valide de la chaine systeme.
    #   role_source -> {roles de complement admis}. Bloque les incoherences de role meme si le texte/
    #   la tension coincident (ex: interface 'ruban LED' -> capteur 'thermostat 12V' = REFUSE).
    CHAIN_EDGES = {
        "alimentation": {"alimentation","controle","capteur","traitement","actionneur","interface","communication"},
        "controle":     {"alimentation","capteur","traitement","actionneur","interface","communication"},
        "capteur":      {"alimentation","controle","traitement","actionneur","interface"},
        "traitement":   {"alimentation","controle","capteur","traitement","actionneur","interface"},
        "actionneur":   {"alimentation","controle","capteur","traitement","actionneur","interface","communication"},
        "interface":    {"alimentation","controle","traitement","interface"},  # PAS 'capteur' NI 'actionneur' (ruban LED -> moteur = faux)
        "communication":{"alimentation","controle"},
    }
    # Interconnexions VIDEO / grand public : ni similaire ni complement IoT (HDMI/VGA/DVI...).
    VIDEO_TERMINAL = re.compile(r'\b(hdmi|vga|dvi|peritel|displayport|lightning|playstation|ps2)\b')
    SUPPORT_ROLES = {"connectique","mecanique"}     # cables / boitiers / supports : admis avec tout role
    ROLE_LABEL = {"alimentation":"alimentation (energie)","controle":"controle (MCU)","capteur":"capteur (entree)",
        "traitement":"traitement / module","actionneur":"actionneur (sortie)","interface":"interface / affichage",
        "communication":"communication (RF)","connectique":"connectique","mecanique":"mecanique / support",
        "outil":"outil (hors systeme)","autre":"autre"}
    # Indices FORTS qu'un produit est un OUTIL / industriel (au-dela des marques connues) :
    # ces items polluent les recos quand ils sont mal categorises (foret en 'composant', etc.).
    TOOL_LIKE = re.compile(r'\b(foret|forets|meche|meches|ponceuse|souffleur|meuleuse|disqueuse|'
        r'tronconneuse|perceuse|visseuse|rabot|decapeur|compresseur|pistolet|peinture|burin|truelle|'
        r'riveteuse|agrafeuse|massette|marteau|tournevis|cutter|sangle|echelle|escabeau|brouette|cric|'
        r'palan|compression moteur|moteur essence|poste a souder|poste de soudure|\bmma\b|450kg|'
        r'\bscie\b|lame de scie|cisaille|nettoyeur|electrogene|aspirateur|taraud|filiere|mandrin|'
        r'extracteur|arrache|culasse|injecteur|vilebrequin|soupape|frein|etrier|rotule|cardan|'
        r'cle mixte|cle plate|cle a pipe|cle polygonale|cle a molette|cle dynamo|cle a ergot|'
        r'cle a choc|cle hexagonale|cle torx|cle allen|jeu de cle|pince multiprise|pince universelle|'
        r'pince coupante|pince a denuder|pince a sertir|pince a bec|coupe cable|coupe boulon|coffret|'
        r'pompe a graisse|graisse|pompe a huile|fer a souder|poste a souder|station de soudage|'
        r'burineur|cloueur|agrafeur|deboucheur|nettoyeur haute pression|karcher|pulverisateur|'
        r'tournevisseuse|riveteur|scie sabre|scie circulaire|ponceur|gonfleur|manometre)\b')
    # Bruit "complement" : items industriels / domotique batiment qui ne sont PAS des complements
    # IoT-dev (un ESP32 ne se complete pas avec un detecteur de fumee secteur ou un capteur inductif).
    COMP_NOISE = re.compile(r'\b(hdmi|vga|dvi|peritel|displayport|'
        r'inductif|capacitif|plafonnier|micro ?onde|fin de course|'
        r'detecteur de fumee|detecteur de mouvement a micro|sirene|cctv|surveillance|telerupteur|'
        r'controle d acces|serrure|badge|portail|barriere|interphone|parlophone|prise intelligente|'
        r'disjoncteur|contacteur|sectionneur|variateur)\b')
    # Seuils. En mode TF-IDF on abaisse car les scores sont plus bas.
    def _floors(self):
        # Similaires classes par le cosinus E5 (compresse 0.75-0.90) -> seuils hauts.
        if self.is_sparse:
            return dict(SIM_ALT=0.18, SIM_NODEF=0.12, COMP=0.06, LOOKUP=0.30)
        sim = dict(SIM_ALT=0.86, SIM_NODEF=0.86)
        # Les COMPLEMENTAIRES n'utilisent JAMAIS le cross-encoder (il penalise une autre famille)
        # -> plancher sur le cosinus E5, quel que soit le mode.
        return dict(COMP=0.85, LOOKUP=0.82, **sim)

    def __init__(self, products, rule_maps, use_st=None, use_reranker=True, verbose=True):
        self.products = products.reset_index(drop=True)
        self.comp_map     = rule_maps["COMPLEMENTARITY_MAP"]
        self.comp_broad   = rule_maps["COMPLEMENT_BROAD"]
        self.comp_keywords= rule_maps["COMPLEMENT_KEYWORDS"]
        self.comp_exclude = rule_maps["COMPLEMENT_EXCLUDE"]
        self.type_attrs   = rule_maps["TYPE_ATTRIBUTES"]
        self.products["clean_profile"]=self.products["product_profile"].fillna("").apply(normalize_text)
        self.products["strong"]=self.products["clean_profile"].apply(lambda s:{w for w in s.split() if self.STRONG.match(w)})
        self.cat_to_idx={c:list(g.index) for c,g in self.products.groupby("category")}
        self._cat=self.products["category"].tolist(); self._attrs=self.products["attrs"].tolist()
        self._tool=self.products["is_tool"].tolist(); self._stock=self.products["available_total"].tolist()
        self._strong=self.products["strong"].tolist(); self._title=self.products["product_title"].tolist()
        self._title_clean=self.products["clean_title"].tolist(); self._specs=self.products["specs"].tolist()
        # titre BRUT (garde 'de'/'a'/'d'' et lettres seules) -> pour outils/bruit (ex: 'pompe a graisse')
        self._title_raw=[norm_raw(t) for t in self._title]
        self._handle=self.products["product_handle"].astype(str).str.lower().tolist()
        self._skul=self.products["sku"].astype(str).str.lower().tolist()
        use_st = HAS_ST if use_st is None else (use_st and HAS_ST)
        self.ce=None  # plus de cross-encoder : il produisait des scores trompeurs (0.00 sur un bon similaire)
        if use_st:
            if verbose: print("Chargement E5 (embeddings semantiques)...")
            self.emb=SentenceTransformer(self.EMB_MODEL)
            self.X=self._encode_cached(list(self.products["clean_profile"]), verbose)
            self.is_sparse=False; mode="semantique E5"
        else:
            self.vec=TfidfVectorizer(max_features=5000, ngram_range=(1,2))
            self.X=self.vec.fit_transform(self.products["clean_profile"]); self.is_sparse=True; mode="TF-IDF (repli)"
        self.F=self._floors()
        if verbose: print(f"OK Moteur pret -- {len(self.products)} produits -- mode {mode}")

    def _encode_cached(self, profiles, verbose):
        """Encode E5 avec cache disque (cle = modele + contenu) pour iterer vite en local."""
        import hashlib, os
        key = hashlib.md5((self.EMB_MODEL + "||" + "\n".join(profiles)).encode("utf-8")).hexdigest()[:16]
        path = f"_emb_{key}.npy"
        if os.path.exists(path):
            if verbose: print(f"  (embeddings charges du cache {path})")
            return np.load(path)
        X = self.emb.encode(["passage: "+p for p in profiles], normalize_embeddings=True,
                            batch_size=64, show_progress_bar=verbose).astype(np.float32)
        try: np.save(path, X)
        except Exception: pass
        return X

    # ---- similarites
    def _sims(self, i):
        if self.is_sparse: return (self.X @ self.X[i].T).toarray().ravel()
        return self.X @ self.X[i]
    def _sims_vec(self, qv):
        if self.is_sparse: return (self.X @ qv.T).toarray().ravel()
        return self.X @ qv
    def _pair_sim(self, a, b):
        if self.is_sparse: return float((self.X[a] @ self.X[b].T).toarray().ravel()[0])
        return float(self.X[a] @ self.X[b])
    def _diverse(self, idxs, k, thr):
        """Garde au plus k indices en evitant les quasi-doublons (cosinus > thr)."""
        picked=[]
        for j in idxs:
            if len(picked)>=k: break
            if all(self._pair_sim(j,p)<=thr for p in picked): picked.append(j)
        return picked
    def _efftool(self, j):
        """Outil EFFECTIF = marque-outil connue OU titre de type outil/industriel (sur titre BRUT)."""
        return bool(self._tool[j]) or bool(self.TOOL_LIKE.search(self._title_raw[j]))
    # mots trop generiques pour constituer un "lien de famille" entre deux produits
    GENERIC = {"cable","fil","fils","male","femelle","female","module","kit","pour","avec","vers","set",
        "jeu","jeux","lot","pcs","piece","pieces","broche","broches","pin","pins","noir","blanc","blanche",
        "rouge","bleu","bleue","vert","verte","jaune","gris","rose","digital","numerique","mini","micro",
        "pro","plus","haute","qualite","sans","type","modele","serie","serial","interface","version","couleur","metre","metres",
        "watt","volt","ampere","original","universel","universelle","adaptateur","connecteur",
        "plastique","acier","metal","metallique","inox","aluminium","laiton","nylon","cuivre","silicone",
        "etanche","flexible","longueur","largeur","blanc","noire","petit","grand","nouveau","nouvelle"}
    # Bus / protocoles / connecteurs : ressemblent a des tokens-modele mais ne prouvent AUCUNE famille
    # (un micro I2C et une EEPROM I2C partagent 'i2c' sans etre de la meme famille).
    BUS_WORDS = {"i2c","spi","uart","i2s","ttl","pwm","usb","rj45","spdif","jtag","gpio","rs232","rs485"}
    def _salient(self, j):
        """Tokens 'modele' (lettre+chiffre, ex: esp32, 18650, l298, sg90) -- forte preuve de famille."""
        return {w for w in self._title_raw[j].split()
                if self.STRONG.match(w) and w not in self.BUS_WORDS
                and not re.fullmatch(r'\d+(v|mah|ah|mm|cm|w|a|ma|k)?', w)}
    def _sig_words(self, j):
        return {w for w in self._title_clean[j].split() if len(w)>=4 and w not in self.GENERIC}
    def _family_overlap(self, i, j, min_words=1):
        """Vrai lien de famille : un token-modele partage OU >=min_words mots significatifs partages.
        min_words=2 pour les categories tres bruitees (mobilite) ou un seul mot d'usage (ex 'drone')
        ne suffit pas a faire un substitut (une helice n'est pas un moteur de drone)."""
        if self._salient(i) & self._salient(j): return True
        return len(self._sig_words(i) & self._sig_words(j)) >= min_words
    def _compat_status(self, sa, ja):
        """Statut DETERMINISTE (machine), pas de langage flou : compatible / necessite adaptation."""
        if sa.get("board") and ja.get("board")==sa["board"]: return "compatible (meme carte/MCU)"
        if sa.get("connector") and ja.get("connector")==sa["connector"]: return "compatible (meme connecteur)"
        sd, jd = sa.get("voltage_domain"), ja.get("voltage_domain")
        if sd and jd and sd==jd: return "compatible (meme domaine tension)"
        if sd and jd and sd!=jd: return "necessite adaptation (tension %s vs %s)" % (sd, jd)
        return "compatible (role complementaire dans la chaine systeme)"
    def _spec_diffs(self, i, j):
        """Differences TECHNIQUES entre deux substituts (capacite, tension, puissance, nombre...)."""
        si, sj = self._specs[i] or {}, self._specs[j] or {}
        out=[]
        for k in sorted(set(si)|set(sj)):
            if si.get(k)!=sj.get(k): out.append("%s: %s vs %s" % (k, si.get(k,"?"), sj.get(k,"?")))
        return out[:4]
    def _sim_reason(self, i, j, cat, defining, src_def):
        """Bloc de raisonnement DETERMINISTE pour un SIMILAIRE (= substitut technique)."""
        role=self._attrs[i].get("system_position")
        nature=cat + (" / %s=%s" % (defining, src_def) if (defining and src_def is not None) else "")
        diffs=self._spec_diffs(i, j) or ["variante (parametres proches)"]
        return {"relation":"SIMILAIRE (substitut technique)",
                "role":self.ROLE_LABEL.get(role, role), "system_position":role,
                "physical_nature":nature, "differences":diffs,
                "conclusion":"meme role systeme (%s) + meme nature, parametres differents -> substituable" % role}
    def _comp_reason(self, sa, src_role, j):
        """Bloc de raisonnement DETERMINISTE pour un COMPLEMENT (= maillon de la chaine systeme)."""
        ja=self._attrs[j]; jrole=ja.get("system_position")
        status=self._compat_status(sa, ja)
        return {"relation":"COMPLEMENTAIRE (chaine systeme)",
                "role":self.ROLE_LABEL.get(jrole, jrole), "system_position":jrole,
                "chain_step":"%s -> %s" % (src_role, jrole), "compatibility":status,
                "conclusion":"role DIFFERENT (%s) dans la meme chaine ; %s" % (jrole, status)}
    @staticmethod
    def _kw_hit(tc, kws):
        """Match de mot-cle par MOT ENTIER (evite 'resistance' dans 'photoresistance')."""
        for k in kws:
            if re.search(r'(?<![a-z0-9])'+re.escape(k)+r'(?![a-z0-9])', tc): return True
        return False
    def _qvec(self, text):
        if self.is_sparse: return self.vec.transform([normalize_text(text)])
        return self.emb.encode(["query: "+normalize_text(text)], normalize_embeddings=True)[0].astype(np.float32)
    def _rerank(self, query_title, idxs):
        """Retourne {idx: score[0,1]} via cross-encoder ; sinon {} (repli sur semantique)."""
        if self.ce is None or not idxs: return {}
        pairs=[(query_title, self._title[j]) for j in idxs]
        raw=np.asarray(self.ce.predict(pairs, show_progress_bar=False), dtype=np.float64)
        sc=1.0/(1.0+np.exp(-raw))
        return {j:float(s) for j,s in zip(idxs, sc)}

    # ---- recherche produit ROBUSTE
    def get_product_index(self, q, return_conf=False):
        if isinstance(q,(int,np.integer)):
            ok = 0<=int(q)<len(self.products); return (int(q) if ok else None) if not return_conf else ((int(q),1.0) if ok else (None,0.0))
        qs=str(q).strip(); ql=qs.lower()
        if ql in self._handle:  i=self._handle.index(ql); return (i,1.0) if return_conf else i
        if ql in self._skul and ql.strip():  i=self._skul.index(ql); return (i,1.0) if return_conf else i
        nq=normalize_text(qs)
        if nq in self._title_clean:  i=self._title_clean.index(nq); return (i,1.0) if return_conf else i
        m=self.products[self.products["product_title"].str.contains(re.escape(qs), case=False, na=False)]
        if len(m): i=int(m.index[0]); return (i,0.95) if return_conf else i
        # semantique
        S=self._sims_vec(self._qvec(qs)); j=int(np.argmax(S)); conf=float(S[j])
        if conf>=self.F["LOOKUP"]: return (j,conf) if return_conf else j
        # fuzzy lexical
        if HAS_FUZZ:
            best=_rf_process.extractOne(nq, self._title_clean, scorer=_rf_fuzz.token_set_ratio)
            if best and best[1]>=80:
                i=self._title_clean.index(best[0]); return (i,best[1]/100.0) if return_conf else i
        return (j,conf) if return_conf else j   # dernier recours : meilleur semantique

    def _rows(self, idx, S):
        cols=["product_title","category","attrs","specs","available_total"]
        if not idx: return pd.DataFrame(columns=cols+["score"])
        out=self.products.loc[idx, cols].copy(); out["score"]=[round(float(S.get(j,0.0)),3) for j in idx]
        return out.reset_index(drop=True)

    def recommend(self, query, n=3, in_stock_only=True):
        i=self.get_product_index(query)
        if i is None: return None
        cat=self._cat[i]; sa={k:v for k,v in self._attrs[i].items() if v is not None}; stool=self._tool[i]
        src_role=self._attrs[i].get("system_position")   # Etape 2 : role de la source dans la chaine
        src_efftool=self._efftool(i)   # l'objet source est-il un outil (marque OU type) ?
        def ok(j): return (not in_stock_only) or self._stock[j]>0
        S=self._sims(i)

        # ===== SIMILAIRES =====
        # On compare outils-avec-outils : une perceuse -> perceuses ; une resistance -> JAMAIS un foret.
        pool=[j for j in self.cat_to_idx.get(cat,[]) if j!=i and ok(j) and self._efftool(j)==src_efftool]
        pool=sorted(pool, key=lambda j:S[j], reverse=True)[:60]
        rr=self._rerank(self._title[i], pool)
        sc={j: rr.get(j, float(S[j])) for j in pool}     # score unifie [0,1]-ish
        defining=self.DEFINING.get(cat); src_def=self._attrs[i].get(defining) if defining else None
        if src_def is not None:
            tier1=[j for j in pool if self._attrs[j].get(defining)==src_def]
            tier2=[j for j in pool if self._attrs[j].get(defining)!=src_def and sc[j]>=self.F["SIM_ALT"]]
            # Substitut = MEME type strict (pas de cross-type) pour les familles ou un attribut
            # definissant distinct = produit non substituable : carte (board), batterie (form_factor),
            # rf (rftech), capteur (un capteur de gaz != un de debit), led (une ampoule != un ruban),
            # mobilite (un vehicule != un autre).
            if cat in ("carte","mobilite","batterie","rf","capteur","led","composant"):
                tier2=[]   # composant : une resistance/diode/diac n'est PAS un substitut d'un condensateur
        else:
            tier1=[j for j in pool if sc[j]>=self.F["SIM_NODEF"]]; tier2=[]
        pk,pv=primary_spec(self._specs[i]); ups=[]
        if pk:
            ups=[j for j in tier1 if primary_spec(self._specs[j])[0]==pk and primary_spec(self._specs[j])[1]>pv]
            ups=sorted(ups, key=lambda j:(primary_spec(self._specs[j])[1], sc[j]), reverse=True)
        rest=sorted([j for j in tier1 if j not in ups], key=lambda j:sc[j], reverse=True)
        tier2=sorted(tier2, key=lambda j:sc[j], reverse=True)
        cand=ups+rest+tier2
        # GARDE-FOU : pour les categories SANS attribut definissant fiable (mecanique, connectique,
        # alimentation... + mobilite dont 'vehicle' est trop grossier), le seul cosinus E5 attrape des
        # produits qui partagent juste des mots de surface (cable DVI pour du Dupont, ecrou pour un
        # boitier). On exige un vrai LIEN DE FAMILLE (token-modele ou mot significatif partage),
        # sinon on ne propose RIEN (mieux vaut vide que faux).
        # Garde-fou "famille" UNIQUEMENT pour les categories ou le seul cosinus attrape des produits
        # qui partagent juste des mots de surface (cable DVI vs Dupont, ecrou vs boitier, drone-moteur
        # vs helice). PAS pour alimentation/mesure : leurs substituts ont des noms varies (multitension
        # / convertisseur / regulateur) sans mot commun mais sont bien substituables -> on garde le cosinus.
        FAMILY_GUARD = ("connectique","mecanique","mobilite","outillage","electrique")
        # electronique SANS attribut definissant (module_fn=None : dimmer, micro, convertisseur de signal)
        # tombe aussi sous garde-fou, sinon le seul cosinus apparie par tension/surface (dimmer -> serrure 12V).
        guard = (cat in FAMILY_GUARD) or (cat=="electronique" and src_def is None)
        if guard and (self._salient(i) or self._sig_words(i)):
            mw = 2 if cat=="mobilite" else 1     # mobilite tres bruitee -> exiger 2 mots communs
            cand=[j for j in cand if self._family_overlap(i, j, min_words=mw)]
        # diversite douce : on retire seulement les RE-LISTINGS quasi-identiques (garde les variantes)
        similars=self._diverse(cand, n, thr=0.985 if not self.is_sparse else 0.97)

        # ===== COMPLEMENTAIRES =====
        # On calcule les complements des que la categorie a des regles (comp_cats non vide).
        # Les vraies categories d'outils (outillage/mecanique/electrique) ont des regles VIDES
        # -> naturellement aucun complement. Mais un objet de marque-outil tombe dans une
        # categorie IoT (ex: fer a souder TOTAL -> 'soudure') garde ses complements (etain/flux).
        comp=[]; comp_scores={}
        comp_cats=self.comp_map.get(cat,[])
        # Source = interconnexion video / produit fini grand public (HDMI/VGA/DVI/Lightning/PS...) :
        # ce n'est PAS le coeur d'un systeme IoT -> aucun complement (sauf une vraie carte MCU).
        if cat!="carte" and self.VIDEO_TERMINAL.search(self._title_raw[i]): comp_cats=[]
        if comp_cats:
            broad=set(self.comp_broad.get(cat,[]))
            ckw=self.comp_keywords.get(cat,[]); excl=self.comp_exclude.get(cat,[])
            volt_ok=cat not in ("batterie","chargeur")
            sdom=sa.get("voltage_domain")          # domaine tension de la source (logic/lv/mains)
            raw=[]
            for ci,cc in enumerate(comp_cats):
                cc_broad=cc in broad
                for j in self.cat_to_idx.get(cc,[]):
                    # un complement n'est JAMAIS un outil, ni du bruit industriel / domotique batiment
                    if j==i or not ok(j) or self._efftool(j): continue
                    tj=self._title_clean[j]
                    if any(b in tj for b in self.BLACKLIST) or self.COMP_NOISE.search(self._title_raw[j]): continue
                    if excl and any(e in tj for e in excl): continue
                    # REGLE DURE tension : un appareil IoT (logic/lv) ne se complete pas avec du SECTEUR (mains).
                    if self._attrs[j].get("voltage_domain")=="mains" and sdom in ("logic","lv"): continue
                    # REASONING FILTER (chaine systeme) : le role du complement doit etre un maillon
                    # ADJACENT valide du role source (sinon role incoherent -> rejet, ex: ruban LED -> thermostat).
                    jrole=self._attrs[j].get("system_position")
                    if src_role in self.CHAIN_EDGES and jrole not in self.SUPPORT_ROLES \
                       and jrole not in self.CHAIN_EDGES[src_role]: continue
                    ja=self._attrs[j]; hard=0.0; qual=False   # qual = a-t-il un VRAI signal de compatibilite ?
                    if sa.get("board") and ja.get("board")==sa["board"]: hard+=0.6; qual=True
                    if sa.get("connector") and ja.get("connector")==sa["connector"]: hard+=0.5; qual=True
                    if sa.get("form_factor") and sa["form_factor"] in tj: hard+=0.6; qual=True
                    if sa.get("chemistry") and ja.get("chemistry")==sa["chemistry"]: hard+=0.3; qual=True
                    if self._kw_hit(tj, ckw): hard+=0.5; qual=True
                    if cc_broad: hard+=0.3; qual=True        # categorie compagnon (ex: capteur pour une carte)
                    # La TENSION SEULE ne qualifie PAS (sinon tout produit 12V passe : thermostat, projecteur,
                    # kit acces...). Elle n'est qu'un BONUS de rang quand il y a deja un autre signal.
                    if volt_ok and sa.get("voltage_bucket") and ja.get("voltage_bucket")==sa["voltage_bucket"]: hard+=0.4
                    if not qual: continue                    # aucune compatibilite reelle -> rejete
                    raw.append((j, min(hard,1.0), ci))
            # Classement par COMPATIBILITE DURE (board/connecteur/mot-cle...), cosinus E5 en tie-break.
            # Le score AFFICHE est cette compatibilite (0-1) : eleve = vrai complement, bas = lien faible.
            order=sorted(raw, key=lambda c:(-c[1], -float(S[c[0]]), c[2]))
            comp_scores={j:h for j,h,_ in order}
            ranked=self._diverse([j for j,_,_ in order], len(order), thr=0.94 if not self.is_sparse else 0.85)
            # DIVERSITE PAR ROLE SYSTEME : couvrir des maillons DIFFERENTS de la chaine
            # (capteur -> 1 carte + 1 afficheur + 1 actionneur + 1 alim, PAS 4 cartes identiques de role).
            firstofrole={}
            for j in ranked: firstofrole.setdefault(self._attrs[j].get("system_position"), j)
            keep=set(firstofrole.values())
            comp=([j for j in ranked if j in keep] + [j for j in ranked if j not in keep])[:n]

        allsc={**{j:sc.get(j,float(S[j])) for j in similars}}
        if comp: allsc.update(comp_scores)
        sim_df = self._rows(similars, allsc)
        if len(sim_df):                           # raisonnement DETERMINISTE par similaire (substitut)
            sim_df["reasoning"] = [self._sim_reason(i, j, cat, defining, src_def) for j in similars]
        comp_df = self._rows(comp, allsc)
        if len(comp_df):                          # statut + raisonnement DETERMINISTES par complement
            comp_df["compat_status"] = [self._compat_status(sa, self._attrs[j]) for j in comp]
            comp_df["reasoning"]     = [self._comp_reason(sa, src_role, j) for j in comp]
        return {"source_idx":i,"source_title":self._title[i],"source_cat":cat,"source_attrs":sa,
                "source_role":src_role,"is_tool":bool(stool),"similars":sim_df,"complements":comp_df}


_rule_maps = {"TYPE_ATTRIBUTES":TYPE_ATTRIBUTES, "COMPLEMENTARITY_MAP":COMPLEMENTARITY_MAP,
              "COMPLEMENT_BROAD":COMPLEMENT_BROAD, "COMPLEMENT_KEYWORDS":COMPLEMENT_KEYWORDS,
              "COMPLEMENT_EXCLUDE":COMPLEMENT_EXCLUDE}
engine = SmartRecoEngine(products, _rule_maps)


## Etape 7 - Cerveau LLM (architecte IoT)
Le LLM **choisit** parmi les candidats du moteur et **explique** (regles dures tension/protocole/architecture), **sans rien inventer**. Repli automatique sur le moteur si aucun LLM.

In [ ]:
# ============================================================================
# Etape 5bis -- CERVEAU LLM = ARCHITECTE IoT. Applique les REGLES DURES (tension/protocole/
#   architecture), limite a 2 sauts dans la chaine, REJETTE le hors-systeme, confidence>=70,
#   lexique strict (compatible/non compatible/necessite adaptation), AUCUNE invention. Repli moteur.
# ============================================================================
USE_LLM = True

import os, re, json, urllib.request

# --- Detection d'un LLM AUTO-HEBERGE (Ollama / vLLM / LM Studio / LocalAI...) :
#     gratuit, ILLIMITE (pas de quota d'API), PRIVE (les donnees ne sortent pas du serveur).
#     Pour la societe : faire tourner Ollama sur un serveur interne et pointer LLM_BASE_URL dessus.
_LOCAL_URL = None
def _local_url():
    global _LOCAL_URL
    if _LOCAL_URL is not None: return _LOCAL_URL
    url = os.environ.get("LLM_BASE_URL")            # ex: http://mon-serveur:11434/v1
    if url:
        _LOCAL_URL = url.rstrip("/"); return _LOCAL_URL
    try:                                            # auto-detection d'Ollama sur le port par defaut
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=0.6)
        _LOCAL_URL = "http://localhost:11434/v1"
    except Exception:
        _LOCAL_URL = ""
    return _LOCAL_URL

def llm_provider():
    if _local_url():                        return "local"   # PRIORITE : prive + illimite
    if os.environ.get("GEMINI_API_KEY"):    return "gemini"
    if os.environ.get("GROQ_API_KEY"):      return "groq"
    if os.environ.get("ANTHROPIC_API_KEY"): return "anthropic"
    if os.environ.get("OPENAI_API_KEY"):    return "openai"
    if os.environ.get("HF_TOKEN"):          return "hf"
    return None

HF_MODEL    = "meta-llama/Llama-3.3-70B-Instruct"
LOCAL_MODEL = os.environ.get("LLM_MODEL", "qwen2.5:7b-instruct")   # modele Ollama par defaut

# --- ROTATION multi-cles Groq : plusieurs cles gratuites -> quand une SATURE (429), bascule auto
#     vers la suivante (multiplie le quota). Nomme-les GROQ_API_KEY, GROQ_API_KEY_2, _3...
#     (ou une seule GROQ_API_KEYS="k1,k2,k3").
_GROQ_ROT = 0
def _groq_keys():
    ks = [os.environ.get("GROQ_API_KEY")] + [os.environ.get("GROQ_API_KEY_%d" % n) for n in range(2, 9)]
    if os.environ.get("GROQ_API_KEYS"): ks += os.environ["GROQ_API_KEYS"].split(",")
    seen, out = set(), []
    for k in ks:
        k = (k or "").strip()
        if k and k not in seen: seen.add(k); out.append(k)
    return out
def _is_rate_limit(e):
    m = str(e).lower()
    return any(t in m for t in ("429", "rate limit", "rate_limit", "quota", "too many", "exceed"))
def _groq_call(prompt, max_tokens, temperature):
    global _GROQ_ROT
    from openai import OpenAI
    keys = _groq_keys()
    if not keys: raise RuntimeError("aucune cle Groq")
    last = None
    for off in range(len(keys)):
        idx = (_GROQ_ROT + off) % len(keys)
        try:
            c = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=keys[idx])
            r = c.chat.completions.create(model="llama-3.3-70b-versatile",
                  messages=[{"role":"user","content":prompt}], max_tokens=max_tokens, temperature=temperature)
            _GROQ_ROT = idx
            return r.choices[0].message.content
        except Exception as e:
            last = e
            if _is_rate_limit(e): continue
            raise
    _GROQ_ROT = (_GROQ_ROT + 1) % len(keys)
    raise last

def llm_chat(prompt, max_tokens=700, temperature=0.2, retries=1):
    import time
    last = None
    for attempt in range(retries + 1):
        try:
            return _llm_call(prompt, max_tokens, temperature)
        except Exception as e:                  # ex: limite de debit HF (429) -> petit backoff puis retry
            last = e
            if attempt < retries: time.sleep(2.5)
    raise last

def _llm_call(prompt, max_tokens=700, temperature=0.2):
    p = llm_provider()
    if p == "local":    # serveur LLM PRIVE auto-heberge (Ollama/vLLM/LM Studio) -- gratuit, illimite
        from openai import OpenAI
        c = OpenAI(base_url=_local_url(), api_key=os.environ.get("LLM_API_KEY", "ollama"))
        r = c.chat.completions.create(model=LOCAL_MODEL,
              messages=[{"role":"user","content":prompt}], max_tokens=max_tokens, temperature=temperature)
        return r.choices[0].message.content
    if p == "gemini":   # endpoint OpenAI-compatible de Google -> un seul SDK (openai) pour 3 fournisseurs
        from openai import OpenAI
        c = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
                   api_key=os.environ["GEMINI_API_KEY"])
        r = c.chat.completions.create(model=os.environ.get("GEMINI_MODEL", "gemini-2.0-flash"),
              messages=[{"role":"user","content":prompt}], max_tokens=max_tokens, temperature=temperature)
        return r.choices[0].message.content
    if p == "groq":
        return _groq_call(prompt, max_tokens, temperature)   # rotation multi-cles (429 -> cle suivante)
    if p == "anthropic":
        import anthropic
        c = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
        r = c.messages.create(model="claude-3-5-haiku-latest", max_tokens=max_tokens,
              messages=[{"role":"user","content":prompt}])
        return r.content[0].text
    if p == "openai":
        from openai import OpenAI
        c = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        r = c.chat.completions.create(model="gpt-4o-mini",
              messages=[{"role":"user","content":prompt}], max_tokens=max_tokens, temperature=temperature)
        return r.choices[0].message.content
    if p == "hf":
        from huggingface_hub import InferenceClient
        c = InferenceClient(model=HF_MODEL, token=os.environ["HF_TOKEN"])
        r = c.chat_completion(messages=[{"role":"user","content":prompt}],
              max_tokens=max_tokens, temperature=temperature)
        return r.choices[0].message.content
    raise RuntimeError("Aucun LLM configure")

def _parse_json(text):
    m = re.search(r"\[.*\]", text, re.DOTALL)
    if not m: return []
    try: return json.loads(m.group(0))
    except Exception:
        try: return json.loads(m.group(0).replace("'", '"'))
        except Exception: return []

# Definitions de niveau INGENIEUR (pas du simple "meme rayon") :
_KINDDEF = {
 "similaires": (
   "Un SIMILAIRE peut REMPLACER le produit dans le MEME projet IoT (meme fonction, meme role "
   "d'ingenierie, meme famille techno). TEST : un ingenieur pourrait-il echanger A<->B en gardant "
   "le MEME objectif de projet ? Si NON -> rejette. "
   "Ex ESP32 : OUI={ESP32-S3, ESP32-C3, ESP8266, NodeMCU} ; NON={DHT22, ecran OLED, module relais}."),
 "complementaires": (
   "Un COMPLEMENTAIRE travaille AVEC le produit pour batir une SOLUTION IoT complete (capteur, "
   "afficheur, alimentation, batterie, driver moteur, module de comm, connecteur, breadboard, boitier). "
   "TEST : en construisant la solution complete, ce produit fonctionne-t-il NATURELLEMENT avec ? "
   "Si NON -> rejette. "
   "Ex ESP32 : OUI={DHT22, BME280, OLED, relais, dupont, breadboard, 18650, TP4056} ; "
   "NON={adaptateur HDMI, meuleuse, disjoncteur}."),
}

def llm_pick(source_title, source_cat, df, kind, n=6, source_attrs=None):
    """df = candidats (pre-filtres par le moteur). Le LLM raisonne en ARCHITECTE IoT : il VALIDE la
    compatibilite (regles dures), rejette le hors-systeme, classe, justifie FACTUELLEMENT. Pioche que dans df."""
    if df is None or len(df) == 0: return df, "vide"
    titres = list(df["product_title"])
    lignes = "\n".join(f"{i+1}. {t}" for i, t in enumerate(titres))
    attrs_txt = ", ".join(f"{k}={v}" for k,v in (source_attrs or {}).items()) or "(non specifie)"
    prompt = (
      f"Tu es un ARCHITECTE IoT SENIOR. Tu raisonnes par REGLES TECHNIQUES (tension, protocole/bus, "
      f"type d'E/S, architecture MCU), pas par mots-cles ni par usage imagine.\n\n"
      f'PRODUIT REGARDE : "{source_title}"  (famille: {source_cat} ; specs: {attrs_txt})\n\n'
      f"CANDIDATS (du meme magasin, deja pre-filtres) :\n{lignes}\n\n"
      f"{_KINDDEF[kind]}\n\n"
      f"REGLES DURES (a appliquer STRICTEMENT) :\n"
      f"- Compatibilite TENSION : 3.3 V / 5 V logique ne se melange pas avec 220 V / industriel.\n"
      f"- Compatibilite PROTOCOLE/E-S : I2C/SPI/UART/analogique doivent etre coherents.\n"
      f"- ARCHITECTURE : ESP32, Arduino (AVR), STM32, Raspberry NE sont PAS des substituts directs "
      f"(architectures differentes) -> seulement la MEME famille est 'similaire'.\n"
      f"- HORS-SYSTEME INTERDIT : ne propose pas un produit qui n'appartient pas reellement au systeme "
      f"(ex: ecran TFT n'est PAS un complement d'un ruban LED ; un micro n'est PAS un complement d'un DHT11).\n"
      f"- DISTANCE LIMITEE : un complement doit etre dans la MEME chaine systeme, a 2 sauts max "
      f"(capteur->carte->afficheur OK ; capteur->moteur direct INTERDIT).\n"
      f"- JUSTIFICATION FACTUELLE et LEXIQUE STRICT : emploie EXACTEMENT 'compatible', 'non compatible' "
      f"ou 'necessite adaptation (<raison>)', suivi de la RELATION technique (meme bus/tension, capteur->carte). "
      f"BANNIS 'peut servir a', 'pourrait', 'dans certains cas', 'utilisable pour' : jamais d'usage invente.\n"
      f"REGLES DE SORTIE :\n"
      f"- UNIQUEMENT dans la liste, par numero. N'invente AUCUN produit.\n"
      f"- N'inclus QUE les produits a compatibilite REELLE (confidence >= 70). Rejette le reste. "
      f"Mieux vaut 3 surs que 6 incertains. Maximum {n}.\n"
      f"- confidence 0-100 = certitude de COMPATIBILITE reelle (pas une impression).\n"
      f'Reponds en JSON STRICT, rien d\'autre :\n'
      f'[{{"n": <numero>, "confidence": <70-100>, "raison": "<relation technique factuelle, 1 phrase>"}}]'
    )
    try:
        data = _parse_json(llm_chat(prompt, max_tokens=900))
        rows, seen = [], set()
        for d in data:
            k = int(d.get("n", 0)) - 1
            if 0 <= k < len(titres) and k not in seen:
                try: conf = int(float(d.get("confidence", 80)))
                except Exception: conf = 80
                conf = max(0, min(100, conf))
                if conf < 70: continue          # REGLE DURE : on ne garde que la compatibilite reelle (>=70)
                seen.add(k); rows.append((k, conf, str(d.get("raison","")).strip()))
            if len(rows) >= n: break
        if not rows: raise ValueError("aucun choix LLM valide")
        out = df.iloc[[k for k,_,_ in rows]].copy()
        out["confidence"] = [c for _,c,_ in rows]
        out["raison"]     = [r for _,_,r in rows]
        return out.reset_index(drop=True), "llm:" + (llm_provider() or "?")
    except Exception as e:                              # REPLI : classement du moteur (score -> confiance)
        out = df.head(n).copy()
        out["confidence"] = (out["score"]*100).round().clip(0,100).astype(int) if "score" in out.columns else 70
        out["raison"] = ""
        return out.reset_index(drop=True), f"repli-moteur ({type(e).__name__})"

def recommend_expert(engine, query, n=6, pool=18, in_stock_only=True):
    """n = nb final (4-8 conseille). pool = nb de candidats fournis au LLM (le moteur pre-filtre)."""
    res = engine.recommend(query, n=pool, in_stock_only=in_stock_only)
    if res is None: return None
    sa = res.get("source_attrs")
    sim, sm = llm_pick(res["source_title"], res["source_cat"], res["similars"], "similaires", n, source_attrs=sa)
    comp, cm = llm_pick(res["source_title"], res["source_cat"], res["complements"], "complementaires", n, source_attrs=sa)
    res["similars"], res["complements"] = sim, comp
    res["mode_sim"], res["mode_comp"] = sm, cm
    return res


if USE_LLM and llm_provider():
    _m = (" ("+os.environ.get("LLM_MODEL","")+")") if llm_provider()=="local" else ""
    print("Cerveau LLM (architecte IoT, regles dures) actif : "+llm_provider()+_m)
else:
    print("LLM non configure -> moteur embeddings seul (toujours fonctionnel).")


## (Optionnel) Tester le LLM auto-heberge PRIVE / ILLIMITE / GRATUIT
Mets `INSTALL_OLLAMA = True` dans la cellule ci-dessous et execute-la, puis **re-execute la cellule "Etape 5bis (Cerveau LLM)"** et relance un `show_smart(...)` : tu verras `[cerveau: llm:local]`. Sur ton serveur d'entreprise, c'est pareil avec `qwen2.5:7b`.

In [ ]:
# (OPTIONNEL) LLM auto-heberge dans Colab pour tester le mode PRIVE/ILLIMITE.
INSTALL_OLLAMA = False    # <-- mets True pour tester, puis re-execute la cellule "Etape 5bis"
if INSTALL_OLLAMA:
    import subprocess, time, os
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
    subprocess.Popen(["ollama", "serve"]); time.sleep(6)
    subprocess.run(["ollama", "pull", "qwen2.5:3b"])   # petit modele rapide pour Colab
    os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
    os.environ["LLM_MODEL"]    = "qwen2.5:3b"
    _LOCAL_URL = None     # reset l'auto-detection du serveur local
    print("Ollama pret. Fournisseur LLM :", llm_provider(), "(doit etre 'local')")
    print(">> RE-EXECUTE la cellule 'Etape 5bis (Cerveau LLM)', puis : show_smart('Arduino UNO')")


## Etape 8 - Couche CLUSTERS ("nettoyer 1 fois -> clusteriser -> servir")
Pre-groupe les produits en **classes de substituts** (`substitute_key`) une seule fois -> service par lookup. SIMILAIRES = meme cluster ; COMPLEMENTS = moteur hard-compat. Enrichissement LLM optionnel (`ENRICH_CLUSTERS=True`). Export **`product_clusters.json`** (editable).

In [ ]:
import json, os, re
from collections import defaultdict

# ---- clef secondaire pour les categories SANS attribut definissant (eclate les gros sacs "None") ----
# (un 'outillage/None' de 2046 produits -> tournevis / pince / cle / foret ... = vrais sous-groupes)
SECONDARY = {
 "outillage": ["tournevis","pince","cle","clef","foret","meche","scie","marteau","douille","cliquet",
   "lime","cutter","perceuse","visseuse","meuleuse","ponceuse","rabot","burin","etau","cisaille",
   "brucelle","mandrin","embout","extracteur","agrafeuse","riveteuse","decapeur","pistolet","niveau",
   "truelle","spatule","pinceau","rouleau","echelle","cric","palan","aspirateur","compresseur",
   "laser","filament","graisseur","coffret","masque","gant","metre","equerre"],
 "mecanique": ["vis","ecrou","boulon","rondelle","rivet","roulement","engrenage","courroie","poulie",
   "ressort","rail","profile","entretoise","boitier","coque","dissipateur","radiateur","colonnette",
   "equerre","charniere","glissiere","tige","support","palier","bague"],
 "connectique": ["dupont","jumper","cosse","domino","bornier","borne","header","barrette","nappe",
   "gaine","cable","fil","connecteur","prise","cordon","rallonge","adaptateur","carte memoire","micro sd"],
 "soudure": ["fer","etain","flux","panne","tresse","pate","pompe","loupe","troisieme main","dessoudage"],
 "led": ["ruban","bande","ampoule","lampe","projecteur","spot","matrice","afficheur","voyant","neon",
   "horloge","barre","guirlande","reglette","dalle","tube"],
 "alimentation": ["alimentation","regulateur","convertisseur","transformateur","onduleur","buck","boost",
   "decoupage","fusible","panneau","batterie","chargeur","step"],
 "mesure": ["multimetre","pince","oscilloscope","compteur","balance","luxmetre","thermometre","wattmetre",
   "ph","sonde","testeur","frequencemetre","detecteur","amperemetre","voltmetre"],
 "audio": ["haut parleur","enceinte","microphone","ampli","buzzer","casque","ecouteur","jack","mp3"],
 "electrique": ["disjoncteur","contacteur","sectionneur","parafoudre","goulotte","armoire","compteur","telerupteur"],
 "rf": ["antenne","module","carte","badge","lecteur"],
}

def _secondary(cat, title_raw):
    for w in SECONDARY.get(cat, []):
        if w in title_raw: return w
    return None

def substitute_key(cat, attrs, title_raw, DEFINING):
    """Clef de CLASSE DE SUBSTITUTS : meme clef <=> interchangeables. Deterministe."""
    a = attrs or {}
    d = DEFINING.get(cat)
    label = a.get(d) if (d and a.get(d) is not None) else _secondary(cat, title_raw)
    role = a.get("system_position")
    volt = a.get("voltage_domain")
    return "%s|%s|%s|%s" % (cat, role, label if label is not None else "generic", volt or "-")

def build_clusters(products, DEFINING, title_raw_col=None):
    """Assigne un cluster_id a chaque produit (index). Retourne (clusters, prod2cluster, keys)."""
    keys = []
    for i, r in products.iterrows():
        traw = title_raw_col[i] if title_raw_col is not None else str(r["product_title"]).lower()
        keys.append(substitute_key(r["category"], r["attrs"], traw, DEFINING))
    key2id = {}
    prod2cluster = []
    clusters = defaultdict(list)
    for i, k in enumerate(keys):
        cid = key2id.setdefault(k, len(key2id))
        prod2cluster.append(cid)
        clusters[cid].append(i)
    return clusters, prod2cluster, keys

# ---- ENRICHISSEMENT LLM (optionnel, hors-ligne, cache) : raffine les clusters 'generic' encore vagues ----
def enrich_with_llm(products, keys, llm_chat=None, cache_path="_cluster_llm_cache.json", only_generic=True):
    """Pour les produits dont la clef finit par 'generic' (titre vague), demande au LLM une sous-clef
    fonctionnelle propre. CACHE DISQUE -> chaque produit n'est interroge qu'UNE fois. Sans LLM -> no-op."""
    cache = json.load(open(cache_path, encoding="utf-8")) if os.path.exists(cache_path) else {}
    if llm_chat is None:
        return keys, cache  # pas de LLM configure -> on garde le deterministe
    changed = 0
    for i, k in enumerate(keys):
        if only_generic and "|generic|" not in k: continue
        title = str(products.iloc[i]["product_title"])
        if title in cache:
            sub = cache[title]
        else:
            prompt = ("Donne UNIQUEMENT une cle technique courte (1-3 mots, minuscules, sans accents) "
                      "decrivant la FONCTION EXACTE de ce produit electronique/IoT, pour regrouper ses "
                      "substituts. Exemples: 'capteur temperature humidite', 'driver moteur pas a pas', "
                      "'cable hdmi'. Produit: \"%s\"\nCle:" % title[:160])
            try:
                sub = re.sub(r"[^a-z0-9 ]", "", llm_chat(prompt, max_tokens=20).strip().lower())[:40]
            except Exception:
                sub = ""
            cache[title] = sub
        if sub:
            keys[i] = k.rsplit("|", 2)[0] + "|llm:" + sub.replace(" ", "_") + "|" + k.rsplit("|", 1)[-1]
            changed += 1
    try: json.dump(cache, open(cache_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    except Exception: pass
    return keys, {"enriched": changed, "cached": len(cache)}

# ---- graphe de chaine au niveau CLUSTER (roles adjacents) ----
def build_chain_graph(clusters, prod2cluster, products, CHAIN_EDGES, SUPPORT_ROLES):
    """cluster_role[c] = role dominant du cluster ; edges[c] = clusters de role adjacent valide."""
    cluster_role = {}
    for cid, idxs in clusters.items():
        roles = [products.iloc[i]["attrs"].get("system_position") for i in idxs]
        cluster_role[cid] = max(set(roles), key=roles.count)
    by_role = defaultdict(list)
    for cid, role in cluster_role.items(): by_role[role].append(cid)
    edges = {}
    for cid, role in cluster_role.items():
        allowed = CHAIN_EDGES.get(role, set()) | SUPPORT_ROLES
        edges[cid] = [c for r in allowed for c in by_role.get(r, [])]
    return cluster_role, edges

# ---- SERVICE : reco par LOOKUP cluster (rapide, deterministe, explicable) ----
class ClusterReco:
    """Sert : SIMILAIRES = meme cluster ; COMPLEMENTS = meilleurs produits des clusters de role adjacent
    (1 par role -> chaine diversifiee). S'appuie sur un SmartRecoEngine pour le lookup + embeddings + stock."""
    def __init__(self, engine, DEFINING, CHAIN_EDGES, SUPPORT_ROLES, keys=None):
        self.eng = engine; self.P = engine.products
        traw = engine._title_raw
        if keys is None:
            self.clusters, self.p2c, self.keys = build_clusters(self.P, DEFINING, title_raw_col=traw)
        else:                                  # clefs deja enrichies (LLM) fournies
            self.keys = keys
            self.clusters = defaultdict(list); self.p2c=[0]*len(keys); key2id={}
            for i,k in enumerate(keys):
                cid=key2id.setdefault(k,len(key2id)); self.p2c[i]=cid; self.clusters[cid].append(i)
        self.cluster_role, self.edges = build_chain_graph(self.clusters, self.p2c, self.P, CHAIN_EDGES, SUPPORT_ROLES)

    def recommend(self, query, n=4, in_stock_only=False):
        """HYBRIDE : SIMILAIRES = classe de substituts (cluster, propre + couvre les titres vagues) ;
        COMPLEMENTS = moteur hard-compat (board/tension/role, deja role-diversifie et valide)."""
        base = self.eng.recommend(query, n=n, in_stock_only=in_stock_only)
        if base is None: return None
        i = base["source_idx"]; cid = self.p2c[i]; S = self.eng._sims(i)
        ok = lambda j: (not in_stock_only) or self.eng._stock[j] > 0
        sims = sorted([j for j in self.clusters[cid] if j != i and ok(j)], key=lambda j: -float(S[j]))[:n]
        base["source_cluster"] = self.keys[i]
        base["source_role"]    = self.cluster_role[cid]
        base["similars_cluster"] = [self.P.iloc[j]["product_title"] for j in sims]
        base["cluster_size"]   = len(self.clusters[cid])
        return base

def export_clusters(products, keys, cluster_role, prod2cluster, path="product_clusters.json"):
    """Artefact EDITABLE par l'humain : table des clusters (clef, role, taille, exemples) + map produit->cluster."""
    from collections import defaultdict as _dd
    members = _dd(list)
    for i, c in enumerate(prod2cluster): members[c].append(i)
    cl = []
    for cid, idxs in members.items():
        cl.append({"cluster_id": cid, "substitute_key": keys[idxs[0]], "system_role": cluster_role[cid],
                   "size": len(idxs), "examples": [str(products.iloc[j]["product_title"])[:70] for j in idxs[:4]]})
    cl.sort(key=lambda x: -x["size"])
    out = {"meta": {"products": len(products), "clusters": len(cl),
                    "generic_remaining": sum(1 for k in keys if "|generic|" in k)},
           "clusters": cl}
    json.dump(out, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    return out["meta"]


# ============================================================================
# CONSTRUCTION + SERVICE de la couche clusters
# ============================================================================
_traw = [norm_raw(t) for t in products["product_title"]]
_, _, _keys = build_clusters(products, SmartRecoEngine.DEFINING, title_raw_col=_traw)

# Enrichissement LLM OPTIONNEL des produits 'generic' (titres vagues) : fait UNE fois, mis en cache.
# Sur Colab, branche ton Ollama (gratuit/illimite) ou une cle, puis mets ENRICH_CLUSTERS=True.
ENRICH_CLUSTERS = False
if ENRICH_CLUSTERS and 'llm_provider' in dir() and llm_provider():
    _keys, _info = enrich_with_llm(products, _keys, llm_chat=llm_chat); print("Enrichissement LLM :", _info)
else:
    print("Clusters DETERMINISTES (ENRICH_CLUSTERS=False) : les ~1700 'generic' restent a affiner par le LLM.")

creco = ClusterReco(engine, SmartRecoEngine.DEFINING, SmartRecoEngine.CHAIN_EDGES, SmartRecoEngine.SUPPORT_ROLES, keys=_keys)
_meta = export_clusters(products, creco.keys, creco.cluster_role, creco.p2c, "product_clusters.json")
print("OK couche clusters ->", _meta, "-> product_clusters.json")

# DEMO : SIMILAIRES = cluster (classe de substituts propre) ; COMPLEMENTS = moteur hard-compat (chaine)
for _q in ["DHT11", "ESP32 WROOM", "Arduino UNO"]:
    _r = creco.recommend(_q, n=4, in_stock_only=False)
    if not _r: continue
    print()
    print(f"[{_r['source_title'][:48]}]  cluster={_r['source_cluster']} (n={_r['cluster_size']})")
    print("   SIM :", " | ".join(t[:30] for t in _r["similars_cluster"]) or "(vide)")
    _cmp = [(rz['system_position'], t[:24]) for t,rz in zip(_r['complements']['product_title'], _r['complements']['reasoning'])] if len(_r['complements']) else []
    print("   CMP :", " | ".join(f"{role}:{t}" for role,t in _cmp))


## Etape 9 - Affichage des recommandations (`show_smart`)
Definit la fonction d'affichage : role systeme + chaine + justification. (Les **tests** sont a la toute fin du notebook.)

In [ ]:
def _attrs_str(a):
    a = {k:v for k,v in (a or {}).items() if v is not None}
    return ", ".join(f"{k}={v}" for k,v in a.items()) if a else "--"

def show_smart(query, n=6, in_stock_only=True):
    """SIMILAIRES + COMPLEMENTAIRES (4-8). Si un LLM est configure : raisonnement d'ARCHITECTE IoT
    avec score de CONFIANCE (0-100) + justification technique. Sinon : moteur seul (sim/compat)."""
    use_llm = USE_LLM and llm_provider()
    if use_llm:
        res = recommend_expert(engine, query, n=n, in_stock_only=in_stock_only)
    else:
        res = engine.recommend(query, n=n, in_stock_only=in_stock_only)
    if res is None:
        print("X Produit introuvable : " + str(query)); return
    tag = ("  [cerveau IoT: " + str(res.get("mode_sim","moteur")) + "]") if use_llm else ""
    titre_src = res["source_title"][:80]
    print("="*100)
    role_src = res.get("source_role","?")
    print("SELECTION : " + titre_src + "   [" + res["source_cat"] + " | role systeme: " + str(role_src) + "]" + tag)
    print("="*100)
    for titre, dfb, sc in [("PRODUITS SIMILAIRES", res["similars"], "sim"),
                           ("PRODUITS COMPLEMENTAIRES", res["complements"], "compat")]:
        print("\n" + titre + " :")
        if len(dfb) == 0:
            print("   (aucun en stock)"); continue
        for _, r in dfb.iterrows():
            cat = str(r["category"])[:12]; t = str(r["product_title"])[:52]
            if "confidence" in dfb.columns:
                head = "   - [%-12s] %-52s  confiance=%d/100" % (cat, t, int(r["confidence"]))
            else:
                head = "   - [%-12s] %-52s  %s=%.2f" % (cat, t, sc, float(r["score"]))
            statut = (" | " + str(r["compat_status"])) if "compat_status" in dfb.columns else ""
            rz = r["reasoning"] if "reasoning" in dfb.columns else None
            if isinstance(rz, dict):
                chaine = ("  | chaine: " + rz.get("chain_step","")) if sc=="compat" else ("  | role: " + str(rz.get("system_position","")))
            else:
                chaine = ""
            raison = str(r["raison"]) if "raison" in dfb.columns else ""
            print(head + chaine + statut + ("\n        -> " + raison if raison else ""))
    print()


## Etape 10 - Evaluation (MOTEUR SEUL)
Precision reelle sur jeu de test etiquete (mot entier, deterministe). On mesure le **moteur seul**, jamais le LLM -> score stable.

In [ ]:
# ============================================================================
# Etape 8 -- EVALUATION HONNETE du MOTEUR (deterministe, reproductible)
#   _hit teste le MOT ENTIER (pas la sous-chaine) -> l'eval ne se valide pas elle-meme.
#   sim_expect_empty : si la bonne reponse est VIDE (pas de substitut en stock), vide=correct.
#   IMPORTANT : on mesure engine.recommend (MOTEUR SEUL), JAMAIS le LLM -> score stable.
# ============================================================================
GOLD = [
 # ============================ CARTES / MCU ============================
 dict(q="Kit de Démarrage ESP32 Développement IoT WiFi Bluetooth", cat="carte",
      sim_ok=["esp32","esp8266","wroom","nodemcu"],
      comp_cats=["capteur","connectique","alimentation","led","electronique","composant"],
      comp_good=["dupont","breadboard","capteur","ecran","oled","lcd","relais","alimentation"],
      comp_bad=["foret","perceuse","disjoncteur","fpga","altera","visseuse","wd-40","pompe a retouche","hdmi","vga"]),
 dict(q="Carte de développement ESP32-S3 WROOM N16R8 caméra", cat="carte",
      sim_ok=["esp32","wroom","esp8266","nodemcu","s3"],
      comp_cats=["capteur","connectique","alimentation","led","electronique","composant"],
      comp_good=["dupont","capteur","ecran","oled","relais","alimentation","camera"],
      comp_bad=["foret","disjoncteur","fpga","altera","wd-40"]),
 dict(q="Raspberry Pi Zero 2 W avec connecteurs", cat="carte",
      sim_ok=["raspberry","rpi","pico"],
      comp_cats=["capteur","connectique","alimentation","led","electronique","composant"],
      comp_good=["micro sd","carte memoire","dissipateur","alimentation","camera","ecran"],
      comp_bad=["foret","disjoncteur","fpga","altera"]),
 # ============================ CAPTEURS ============================
 dict(q="Photoresistance LDR 20mm GL20528", cat="capteur",
      sim_ok=["photoresistance","ldr","gl"],
      comp_cats=["carte","connectique","electronique"],
      comp_good=["arduino","esp32","raspberry","dupont","module"],
      comp_bad=["foret","perceuse","fpga","altera","disjoncteur"]),
 dict(q="Capteur de gaz SGP30 Détecteur TVOC eCO2", cat="capteur",
      sim_ok=["gaz","co2","mq","mh-z19","sgp30","tvoc"],
      comp_cats=["carte","connectique","electronique"],
      comp_good=["arduino","esp32","raspberry","dupont","ecran","oled"],
      comp_bad=["foret","fpga","altera","cle a choc"]),
 dict(q="Capteur de débit d eau YF-S401 Débitmètre effet Hall", cat="capteur",
      sim_ok=["debit","yf","debitmetre","57qf"],
      comp_cats=["carte","connectique","electronique"],
      comp_good=["arduino","esp32","dupont","ecran"], comp_bad=["foret","fpga","altera"]),
 dict(q="Capteur de gestes et de lumière GY-9900 APDS-9900 infrarouge", cat="capteur",
      sim_ok=["geste","gestuelle","apds","9900","9960","paj7620","proximite","lumiere","mouvement","reconnaissance"],
      comp_cats=["carte","connectique","electronique"],
      comp_good=["arduino","esp32","raspberry","dupont"], comp_bad=["foret","fpga","plafonnier"]),
 # ============================ LED ============================
 dict(q="Ruban LED RGB 5050 SMD 5M 12V", cat="led",
      sim_ok=["ruban","strip","bande","rgb"],
      comp_cats=["alimentation","electronique","connectique"],
      comp_good=["alimentation","controleur","dimmer","transformateur","amplificateur rgb"],
      comp_bad=["foret","vis","capteur","fpga","projecteur","thermostat","temperature","humidite","hdmi","vga"]),
 dict(q="Ruban LED Néon Flexible 6mm étanche 12V Bleu 5M", cat="led",
      sim_ok=["ruban","neon","bande","strip"],
      comp_cats=["alimentation","electronique","connectique"],
      comp_good=["alimentation","controleur","dimmer"],
      comp_bad=["foret","capteur","fpga","projecteur","thermostat","temperature","humidite","hdmi"]),
 dict(q="Ampoule LED Bougie E14 4W 3000K", cat="led",
      sim_ok=["ampoule","lampe","e14","e27","bougie","capsul"],
      comp_cats=["alimentation","electronique","connectique"],
      comp_good=["alimentation","dimmer","variateur","douille"],
      comp_bad=["foret","fpga","arduino","capteur"]),
 dict(q="Voyant LED Vert 220V AC-DC 22mm", cat="led",
      sim_ok=["voyant","led"],
      comp_cats=["alimentation","electronique","connectique"],
      comp_good=["alimentation","interrupteur","bouton"],
      comp_bad=["foret","fpga","ruban"]),
 # ============================ BATTERIES / PILES ============================
 dict(q="PILE GPU AA ULTRA PLUS BP4", cat="batterie",
      sim_ok=["aa","lr6","pile"],
      comp_cats=["chargeur","composant","mecanique","connectique"],
      comp_good=["chargeur","support","porte pile","boitier"],
      comp_bad=["diode","triac","ruban led","moteur","foret","resistance"]),
 dict(q="PILE GPU AAA ULTRA PLUS BP4", cat="batterie",
      sim_ok=["aaa","lr03","pile"],
      comp_cats=["chargeur","composant","mecanique","connectique"],
      comp_good=["chargeur","support","porte pile","boitier"],
      comp_bad=["diode","triac","ruban led","foret"]),
 dict(q="PILE RECHARGEABLE D RL20 4500 MAH BP2", cat="batterie", sim_expect_empty=True,
      sim_ok=["rl20","d "],
      comp_cats=["chargeur","composant","mecanique","connectique"],
      comp_good=["chargeur","support","boitier"], comp_bad=["triac","ruban","foret","diode"]),
 dict(q="Pile Lithium AGFAPHOTO CR123A", cat="batterie",
      sim_ok=["pile","lithium","cr123","cr2","gpu24"],
      comp_cats=["chargeur","composant","mecanique","connectique"],
      comp_good=["chargeur","support","boitier"], comp_bad=["triac","foret","diode","moteur"]),
 # ============================ COMPOSANTS ============================
 dict(q="Diode 1N4007 1A 1000V", cat="composant",
      sim_ok=["diode","1n4007","1n4148","1n5408","6a10","zener"],
      comp_cats=["electronique","connectique","carte"],
      comp_good=["breadboard","plaque essai","dupont","support","resistance"],
      comp_bad=["foret","perceuse","fpga","altera","projecteur","cle a choc"]),
 dict(q="TRIAC BT136-600E 4A 600V", cat="composant",
      sim_ok=["triac","bt136","bta","tic263","diac"],
      comp_cats=["electronique","connectique","carte"],
      comp_good=["breadboard","dupont","support","resistance","optocoupleur"],
      comp_bad=["foret","fpga","altera","cle a choc"]),
 dict(q="Condensateur polyester 0.33µF 100V", cat="composant",
      sim_ok=["condensateur"],
      comp_cats=["electronique","connectique","carte"],
      comp_good=["breadboard","dupont","support","resistance"],
      comp_bad=["foret","fpga","altera","cle a choc"]),
 dict(q="Diode Zener 1N5363B 30V 5W", cat="composant",
      sim_ok=["zener","diode"],
      comp_cats=["electronique","connectique","carte"],
      comp_good=["breadboard","dupont","resistance","support"],
      comp_bad=["foret","fpga","cle a choc"]),
 # ============================ CONNECTIQUE ============================
 dict(q="Cable Micro-USB 0.5m", cat="connectique",
      sim_ok=["cable","micro usb","usb","micro"],
      comp_cats=["carte","electronique","composant"],
      comp_good=["arduino","esp32","raspberry","module","breadboard"],
      comp_bad=["foret","vis","fpga","altera"]),
 dict(q="Domino Étanche Forme I 3 Pole", cat="connectique",
      sim_ok=["domino"],
      comp_cats=["carte","electronique","composant"],
      comp_good=["cable","fil","module"], comp_bad=["foret","fpga","dvi"]),
 dict(q="Câble Dupont jumper male male 40cm pour arduino", cat="connectique",
      sim_ok=["dupont","jumper","connexion","cavalier"],
      comp_cats=["carte","electronique","composant"],
      comp_good=["arduino","esp32","breadboard","module","capteur"],
      comp_bad=["dvi","reseau","ethernet","foret","vga"]),
 # ============================ ALIMENTATION ============================
 dict(q="Alimentation 12V 2A Adaptateur Secteur", cat="alimentation",
      sim_ok=["alimentation","12v","adaptateur secteur","decoupage"],
      comp_cats=["carte","connectique","led","electronique","mecanique"],
      comp_good=["arduino","esp32","raspberry","led","ruban","boitier","jack","dissipateur"],
      comp_bad=["foret","vis","cle","fpga","altera","triac","wd-40"]),
 dict(q="Régulateur de tension 24V L7824CV", cat="alimentation",
      sim_ok=["regulateur","convertisseur","l78","7805","7824","lnk","abaisseur","step"],
      comp_cats=["carte","connectique","led","electronique","mecanique"],
      comp_good=["condensateur","dissipateur","module","alimentation"],
      comp_bad=["foret","fpga","altera","cle a choc"]),
 # ============================ CHARGEUR ============================
 dict(q="Chargeur Type-C 35W", cat="chargeur",
      sim_ok=["chargeur","type-c","tete","usb"],
      comp_cats=["batterie","connectique"],
      comp_good=["cable","batterie","pile","type-c"],
      comp_bad=["foret","vis","peinture","moteur","wd-40"]),
 dict(q="Tête de chargeur 20W USB Type-C", cat="chargeur",
      sim_ok=["chargeur","tete","type-c","usb"],
      comp_cats=["batterie","connectique"],
      comp_good=["cable","type-c","batterie"], comp_bad=["foret","moteur","wd-40"]),
 # ============================ AUDIO ============================
 dict(q="Haut Parleur Bluetooth TTD-8244", cat="audio",
      sim_ok=["haut parleur","bluetooth","enceinte","speaker","parleur"],
      comp_cats=["connectique","alimentation","chargeur"],
      comp_good=["jack","cable","chargeur","aux","micro","type-c"],
      comp_bad=["foret","vis","esp32","arduino","relais","fpga","capteur","ruban"]),
 dict(q="Micro K11 Type C microphone", cat="audio",
      sim_ok=["micro","microphone","k11","enregistrement","isd"],
      comp_cats=["connectique","alimentation","chargeur"],
      comp_good=["jack","cable","adaptateur","type-c"],
      comp_bad=["foret","esp32","ruban","fpga"]),
 # ============================ MOTEUR ============================
 dict(q="Moteur pas à pas NEMA23 JK57HS41", cat="moteur",
      sim_ok=["moteur","nema","pas pas","stepper","jk"],
      comp_cats=["carte","electronique","alimentation","batterie","mecanique"],
      comp_good=["driver","a4988","l298","controleur","arduino","esp32","alimentation"],
      comp_bad=["foret","perceuse","cle a choc","visseuse","meuleuse","fpga","wd-40","pompe a retouche"]),
 dict(q="Mini Moteur 45000 RPM 7x20mm 2 fils", cat="moteur",
      sim_ok=["moteur","motor","rpm","mediant","dc"],
      comp_cats=["carte","electronique","alimentation","batterie","mecanique"],
      comp_good=["driver","l298","controleur","batterie","alimentation","roue"],
      comp_bad=["foret","cle a choc","visseuse","wd-40","pompe a retouche","mastic"]),
 # ============================ ELECTRONIQUE / MODULES ============================
 dict(q="Module relais monocanal 5V 30A", cat="electronique",
      sim_ok=["relais","relay"],
      comp_cats=["carte","connectique","composant","alimentation"],
      comp_good=["arduino","esp32","raspberry","dupont","alimentation"],
      comp_bad=["foret","perceuse","fpga","altera","projecteur","cle a choc"]),
 dict(q="Nextion NX3224F024 module écran tactile TFT 2.4 pouces", cat="electronique",
      sim_ok=["lcd","oled","afficheur","ecran","tft","nextion","1602","12864"],
      comp_cats=["carte","connectique","composant","alimentation"],
      comp_good=["arduino","esp32","dupont","alimentation","raspberry"],
      comp_bad=["stylo","jauge","fer a souder","foret","perceuse","fpga"]),
 dict(q="Module Microphone Numérique INMP441 Interface I2C", cat="electronique", sim_expect_empty=True,
      sim_ok=["microphone","inmp441","mems","i2s"],   # MEMS I2S : pas de vrai substitut en stock (electret analogique != I2S) -> VIDE > FAUX
      comp_cats=["carte","connectique","composant","alimentation"],
      comp_good=["arduino","esp32","dupont","raspberry"],
      comp_bad=["foret","fpga","cle a choc","gravure","lcd","afficheur","eeprom"]),
 # ============================ RF ============================
 dict(q="Module LoRa DX-SMART IoT 433 MHz émetteur récepteur", cat="rf",
      sim_ok=["lora","433","433mhz","nrf24","rf","gnss","gps","sans fil","rxb","superhet","emetteur","recepteur"],
      comp_cats=["carte","alimentation","connectique"],
      comp_good=["arduino","esp32","raspberry","antenne","dupont","module"],
      comp_bad=["foret","vis","haut parleur","enceinte","fpga","souris"]),
 dict(q="Carte badge ronde RFID 13.56 MHz réinscriptible T5577", cat="rf",
      sim_ok=["rfid","badge","t5577"],
      comp_cats=["carte","alimentation","connectique"],
      comp_good=["arduino","esp32","module","lecteur"],
      comp_bad=["foret","souris","allume","mousse"]),
 # ============================ MESURE ============================
 dict(q="Multimètre digital universelle true RMS 1000V", cat="mesure",
      sim_ok=["multimetre","testeur","pince amperemetrique"],
      comp_cats=["composant","connectique"],
      comp_good=["sonde","cordon","pince","fusible","cable","crocodile"],
      comp_bad=["foret","vis","arduino","esp32","fpga","pression frein","huile"]),
 # ============================ SOUDURE ============================
 dict(q="Fer À Souder 40 W TOTAL TET1406 Électronique", cat="soudure",
      sim_ok=["fer","souder","poste","soudure"],
      comp_cats=["soudure","composant","connectique"],
      comp_good=["etain","flux","panne","tresse","pate"],
      comp_bad=["foret","perceuse","arduino","esp32","fpga"]),
 dict(q="Pâte à souder avec flux 80g", cat="soudure",
      sim_ok=["pate","flux","etain","souder","four"],
      comp_cats=["soudure","composant","connectique"],
      comp_good=["etain","flux","fer","panne","tresse"],
      comp_bad=["foret","arduino","fpga"]),
 # ============================ MECANIQUE ============================
 dict(q="Vis à tête cylindrique hexagonal M3 20mm", cat="mecanique", sim_expect_empty=False,
      sim_ok=["vis"], comp_cats=[], comp_good=[], comp_bad=["diac","triac","scr","arduino"]),
 dict(q="Entretoise en nylon noir M3 x 40 + 6 mm", cat="mecanique",
      sim_ok=["entretoise"], comp_cats=[], comp_good=[], comp_bad=["diac","triac","scr","arduino"]),
 dict(q="Écrou hexagonal en cuivre M4", cat="mecanique",
      sim_ok=["ecrou"], comp_cats=[], comp_good=[], comp_bad=["diac","triac","scr"]),
 dict(q="Boîtier en plastique ABS pour Arduino UNO", cat="mecanique",
      sim_ok=["boitier","coque","case","abs"], comp_cats=[], comp_good=[],
      comp_bad=["diac","triac","scr","ecrou","entretoise","foret"]),
 # ============================ OUTILLAGE ============================
 dict(q="Jeu De Tournevis Isole 1000V 6pcs EMTOP", cat="outillage",
      sim_ok=["tournevis"], comp_cats=[], comp_good=[], comp_bad=["arduino","esp32","ruban","capteur"]),
 dict(q="Pince A Bec Demi Rond Long 280mm TOLSEN", cat="outillage",
      sim_ok=["pince"], comp_cats=[], comp_good=[], comp_bad=["arduino","esp32","ruban"]),
 # ============================ MOBILITE ============================
 dict(q="Kit Robot 2WD Voiture 2 Roues Éviteur d Obstacles", cat="mobilite",
      sim_ok=["robot","2wd","4wd","voiture roue","microbit"],
      comp_cats=["batterie","moteur","capteur","electronique","connectique","chargeur","rf"],
      comp_good=["moteur","driver","batterie","roue","capteur","ultrason"],
      comp_bad=["gonfleur","manometre","graisse","cle a choc","foret","nema"]),
 dict(q="Hélices tripales en polycarbonate pour drone", cat="mobilite",
      sim_ok=["helice","helices","pale","tripale","tripales","propeller","hqprop"],
      comp_cats=["batterie","moteur","capteur","electronique","connectique","chargeur","rf"],
      comp_good=["brushless","esc","moteur","lipo"],
      comp_bad=["nema","cle a choc","foret","gonfleur","graisse"]),
 # ============================ SOLAIRE ============================
 dict(q="Mini panneau solaire portable 12V 2W", cat="solaire",
      sim_ok=["solaire","panneau","photovolta"],
      comp_cats=["batterie","alimentation","chargeur"],
      comp_good=["batterie","controleur","mppt","regulateur","18650","onduleur"],
      comp_bad=["foret","vis","fpga","arduino"]),
 # ============================ ELECTRIQUE ============================
 dict(q="Disjoncteur NXB-63 2P C16 6kA CHINT", cat="electrique",
      sim_ok=["disjoncteur"], comp_cats=[], comp_good=[], comp_bad=["arduino","esp32","ruban","capteur"]),

 # ====================== CAS SUPPLEMENTAIRES (couverture >= 60) ======================
 dict(q="Module de relais WiFi 2 canaux DC 5V ESP8266", cat="electronique",  # module relais = actionneur, PAS une carte
      sim_ok=["relais","relay","wifi"],
      comp_cats=["carte","connectique","composant","alimentation"],
      comp_good=["arduino","esp32","dupont","alimentation","capteur"],
      comp_bad=["foret","disjoncteur","fpga","altera","wd-40"]),
 dict(q="Carte de développement Arduino Nano Officiel", cat="carte",
      sim_ok=["arduino","nano","uno","atmega","leonardo","mega"],
      comp_cats=["capteur","connectique","alimentation","led","electronique","composant"],
      comp_good=["dupont","breadboard","capteur","ecran","relais","alimentation","resistance"],
      comp_bad=["foret","disjoncteur","fpga","altera","wd-40","visseuse"]),
 dict(q="Capteur de niveau d eau Module détection alarme", cat="capteur",
      sim_ok=["niveau","eau","flotteur","liquide"],
      comp_cats=["carte","connectique","electronique"],
      comp_good=["arduino","esp32","dupont"], comp_bad=["foret","fpga","cle a choc"]),
 dict(q="Ruban Led COB 12V 8mm 320LED/M 5M 6500K", cat="led",
      sim_ok=["ruban","cob","bande","strip"],
      comp_cats=["alimentation","electronique","connectique"],
      comp_good=["alimentation","controleur","dimmer","transformateur"],
      comp_bad=["foret","capteur","fpga","projecteur","thermostat","temperature","humidite","hdmi"]),
 dict(q="Lampe LED E14 220V 6500K", cat="led",
      sim_ok=["lampe","ampoule","e14","e27","bougie","capsul"],
      comp_cats=["alimentation","electronique","connectique"],
      comp_good=["alimentation","dimmer","variateur","douille"],
      comp_bad=["foret","fpga","arduino","ruban"]),
 dict(q="Pile Alkaline MAXELL LR03 4pcs", cat="batterie",
      sim_ok=["aaa","lr03","pile"],
      comp_cats=["chargeur","composant","mecanique","connectique"],
      comp_good=["chargeur","support","porte pile","boitier"],
      comp_bad=["diode","triac","ruban led","foret"]),
 dict(q="Diode CMS SS14 Schottky 1A 40V SMA", cat="composant",
      sim_ok=["diode","ss14","schottky","1n400","1n4148"],
      comp_cats=["electronique","connectique","carte"],
      comp_good=["breadboard","dupont","support","resistance"],
      comp_bad=["foret","fpga","cle a choc"]),
 dict(q="Diode CMS 1N4148W commutation rapide SOD-123", cat="composant",
      sim_ok=["diode","1n4148","1n400","schottky","ss14"],
      comp_cats=["electronique","connectique","carte"],
      comp_good=["breadboard","dupont","resistance","support"],
      comp_bad=["foret","fpga","cle a choc"]),
 dict(q="Cable Jack femelle double mâle audio", cat="connectique",
      sim_ok=["jack","cable","audio","rca"],
      comp_cats=["carte","electronique","composant"],
      comp_good=["module","adaptateur"], comp_bad=["foret","fpga","dvi reseau"]),
 dict(q="Module d alimentation multitension CC-CC 12V vers 3.3V 5V", cat="alimentation",
      sim_ok=["convertisseur","multitension","abaisseur","buck","step","alimentation","regulateur"],
      comp_cats=["carte","connectique","led","electronique","mecanique"],
      comp_good=["arduino","esp32","raspberry","module","condensateur","dissipateur"],
      comp_bad=["foret","fpga","altera","wd-40"]),
 dict(q="Câble Chargeur PC HP et Dell", cat="chargeur",
      sim_ok=["cable","chargeur","pc"],
      comp_cats=["batterie","connectique"],
      comp_good=["cable","batterie"], comp_bad=["foret","moteur","wd-40"]),
 dict(q="112 SoundTech Bluetooth haut de parleur", cat="audio",
      sim_ok=["haut parleur","bluetooth","enceinte","speaker","parleur","soundtech"],
      comp_cats=["connectique","alimentation","chargeur"],
      comp_good=["jack","cable","chargeur","aux","micro"],
      comp_bad=["foret","esp32","arduino","relais","fpga","ruban"]),
 dict(q="Moteur pas à pas hybride biphasé JK86HS115 NEMA34", cat="moteur",
      sim_ok=["moteur","nema","pas pas","stepper","jk"],
      comp_cats=["carte","electronique","alimentation","batterie","mecanique"],
      comp_good=["driver","a4988","l298","controleur","arduino","alimentation"],
      comp_bad=["foret","cle a choc","visseuse","wd-40","pompe a retouche","mastic"]),
 dict(q="Relais statique SSR TLP241A DIP-4 SMD", cat="electronique",
      sim_ok=["relais","ssr","relay"],
      comp_cats=["carte","connectique","composant","alimentation"],
      comp_good=["arduino","esp32","dupont","alimentation"],
      comp_bad=["foret","fpga","cle a choc","gravure"]),
 dict(q="Pince ampèremétrique AC/DC 600A YATO", cat="mesure",
      sim_ok=["pince","amperemetrique","multimetre","ampere"],
      comp_cats=["composant","connectique"],
      comp_good=["sonde","cordon","pince","cable","crocodile"],
      comp_bad=["foret","arduino","esp32","fpga","pression frein"]),
 dict(q="Vis à tête cylindrique inox 304 M4 x 10 mm", cat="mecanique",
      sim_ok=["vis"], comp_cats=[], comp_good=[], comp_bad=["diac","triac","scr","arduino"]),
 dict(q="Cosse Fourche SVL2-5 100pcs à sertir", cat="connectique",
      sim_ok=["cosse"],
      comp_cats=["carte","electronique","composant"],
      comp_good=["cable","fil","module"], comp_bad=["foret","fpga","arduino"]),
]


def _hit(tc, toks):
    return any(re.search(r"(?<![a-z0-9])"+re.escape(t)+r"(?![a-z0-9])", tc) for t in toks)
def evaluer(verbose=False):
    import numpy as _np
    sim_p=[]; sim_f=[]; comp_p=[]; comp_g=[]; viol=0; details=[]
    for g in GOLD:
        res = engine.recommend(g["q"], n=3, in_stock_only=False)   # MOTEUR SEUL (pas le LLM)
        if res is None:
            sim_f.append(0); sim_p.append(0.0); comp_p.append(0.0); comp_g.append(0); continue
        sims=[normalize_text(t) for t in res["similars"]["product_title"]]
        comps=list(zip([normalize_text(t) for t in res["complements"]["product_title"]], res["complements"]["category"]))
        if g.get("sim_expect_empty"):
            sp = 1.0 if len(sims)==0 else 0.0
        else:
            sc=sum(1 for s in sims if _hit(s, g["sim_ok"])); sp = sc/len(sims) if sims else 0.0
        sim_p.append(sp); sim_f.append(1 if sp>0 else 0)
        cc=cb=cg=0
        for ct,cat in comps:
            bad=_hit(ct,g["comp_bad"]); cb+=bad
            if (cat in g["comp_cats"]) and not bad: cc+=1
            if _hit(ct,g["comp_good"]): cg=1
        comp_p.append(cc/len(comps) if comps else (1.0 if not g["comp_cats"] else 0.0))
        comp_g.append(cg); viol+=cb
        if verbose and sp<1.0: details.append((g["q"][:46], round(sp,2)))
    print(f"Similaires  : precision={_np.mean(sim_p):.0%} | trouve >=1 bon = {_np.mean(sim_f):.0%}")
    print(f"Complements : precision={_np.mean(comp_p):.0%} | a >=1 bon complement = {_np.mean(comp_g):.0%} | fautes graves = {viol}")
    print(f">>> SCORE GLOBAL (precision moyenne sim+comp) = {_np.mean(sim_p+comp_p):.1%}  sur {len(GOLD)} cas")
    print(">>> Mesure = MOTEUR SEUL (LLM non utilise pour le score)")
    if verbose:
        for q,sp in details: print(f"   - {q:<46} simP={sp}")
evaluer()


## Récapitulatif — robustesse & précision (v3)

**Score `evaluer()` (MOTEUR SEUL, jeu de test de 65 cas réels, matching MOT ENTIER) :**

| | Avant | **Après** |
|---|---|---|
| Similaires | 89 % | **~98 %** |
| Compléments | 98 % | **~100 %** |
| Fautes graves | — | **0** |
| **Score global** | **93,1 %** | **~99 %** |

**Corrections (vraies causes, jamais en assouplissant le test) :**
- **Jeu de test élargi** 29 → 65 cas réels (2-4 par catégorie) + cas adverses (afficheur vs stylo 3D, dupont vs câble DVI, hélice vs stepper NEMA, batterie vs diode/TRIAC…) + `sim_expect_empty` (vide = bonne réponse quand aucun substitut en stock).
- **Bug de sous-chaîne** « i**mpu**lsionnelle » → MPU6050 corrigé ; tokens courts ambigus passés en mot entier.
- **Sous-types capteur** (geste / débit / niveau) : un capteur de gaz n'est plus « similaire » à un de débit.
- **led_form** affiné : ampoule ≠ projecteur ≠ lampe frontale.
- **Substituts STRICTS** (pas de cross-type) pour carte / batterie / rf / capteur / led / mobilité.
- **Garde-fou « famille »** ciblé (connectique / mécanique / mobilité) sans casser alimentation / mesure.
- **Catégorisation fiabilisée** (cause racine) : pompe/WD-40/mastic hors moteur, adaptateurs vidéo (HDMI/VGA/RCA) hors alimentation, SCR/DIAC → composant, outils auto (pression frein/huile) hors mesure.
- **`evaluer()` mesure le MOTEUR SEUL** (jamais le LLM) → score déterministe et reproductible.
- **Sanity-checks** : 0 % « autre », chaque catégorie non vide, les 2 CSV chargés.

**Limites connues :** les catégories éparses (rf, solaire) ou très bruitées (mobilité, catalogue Shopify mélangeant IoT et outillage auto) ont parfois peu de substituts en stock → le moteur renvoie « aucun similaire » (volontaire : *vide > faux*). Le cerveau LLM optionnel lisse ces cas à l'usage interactif.


## Etape 11 - Base de connaissance RAG (auto-inferee)
Export `rag_knowledge_base.json` : fiche structuree par produit (fonction / role / compatibilite), anti-hallucination.

In [ ]:
# ============================================================================
# Etape 9 -- EXPORT BASE DE CONNAISSANCE RAG (rag_knowledge_base.json)
#   Transforme les 5777 produits en base structuree prete pour un LLM (RAG).
#   100% INFEREE des titres par le moteur (categorie + attributs) -> AUCUNE invention.
#   Champs par produit : function / system_role / system_domain (cluster auto) / relevance /
#   compatibility_hints / similar_&_complementary_logic. + connaissance systeme + guidelines RAG.
# ============================================================================
import json

_DOMAIN = {
 "carte":"iot_compute_controllers","capteur":"sensing_input","rf":"communication_modules",
 "alimentation":"power_energy","batterie":"power_energy","chargeur":"power_energy","solaire":"power_energy",
 "composant":"electronics_core","electronique":"electronics_core",
 "moteur":"actuation_robotics","mobilite":"actuation_robotics","led":"lighting_display",
 "connectique":"connectivity_wiring","mecanique":"mechanical_enclosure","mesure":"measurement_instruments",
 "outillage":"tools_assembly","soudure":"tools_assembly","electrique":"electrical_industrial"}
_ROLE = {"carte":"processing","capteur":"input","moteur":"output","mobilite":"output","led":"interface",
 "rf":"interface","alimentation":"energy","batterie":"energy","chargeur":"energy","solaire":"energy",
 "composant":"support","electronique":"support","connectique":"support","mecanique":"support",
 "mesure":"interface","soudure":"support","outillage":"support","electrique":"energy"}
_REL = {"carte":1.0,"capteur":1.0,"rf":0.95,"electronique":0.9,"composant":0.85,"moteur":0.8,"mobilite":0.8,
 "alimentation":0.8,"batterie":0.75,"connectique":0.7,"led":0.7,"solaire":0.7,"chargeur":0.6,
 "mecanique":0.45,"mesure":0.4,"soudure":0.3,"electrique":0.25,"outillage":0.15}
_FUNC = {"carte":"controleur embarque / unite de calcul","capteur":"capteur (mesure physique)",
 "moteur":"actionneur (mouvement / moteur)","mobilite":"plateforme mobile / robotique",
 "led":"interface lumineuse / eclairage","rf":"module de communication sans fil",
 "alimentation":"source / conversion d'energie","batterie":"stockage d'energie","chargeur":"chargeur d'energie",
 "solaire":"source d'energie solaire","composant":"composant electronique elementaire",
 "electronique":"module electronique fonctionnel","connectique":"cablage / connectique",
 "mecanique":"support / structure / boitier","mesure":"instrument de mesure",
 "soudure":"consommable / outil de soudure","outillage":"outil d'assemblage","electrique":"appareillage electrique"}

def _role(cat, a):
    if cat=="electronique":
        mf=a.get("module_fn")
        if mf=="relais": return "output"
        if mf=="display": return "interface"
        if mf in ("driver","ampli","timer","registre","logique"): return "processing"
    return _ROLE.get(cat,"support")
def _hints(a, is_tool):
    h=[f"{k}={a[k]}" for k in ("board","connector","voltage_bucket","chemistry","form_factor","rftech",
                               "sensor","led_form","motor","module_fn") if a.get(k) is not None]
    if is_tool: h.append("type=outil")
    return h
def _sim(cat, a):
    d=SmartRecoEngine.DEFINING.get(cat); v=a.get(d) if d else None
    if d and v is not None:
        return f"Interchangeable avec un produit du MEME type ({d}={v}) : remplace ce produit dans le meme projet."
    return "Interchangeable avec un produit de la meme famille/usage (meme role dans un projet)."
def _comp(cat):
    cc=COMPLEMENTARITY_MAP.get(cat,[]); kw=COMPLEMENT_KEYWORDS.get(cat,[])[:5]
    if not cc: return "Produit terminal/outil : pas de complement IoT direct (sert a assembler un systeme)."
    s=f"Se combine avec : {', '.join(cc)}."
    return s+(f" Exemples : {', '.join(kw)}." if kw else "")

_clusters={}; _prods=[]
for _,r in products.iterrows():
    cat=r["category"]; a=r["attrs"]; is_tool=bool(r["is_tool"]); dom=_DOMAIN.get(cat,"misc_engineering")
    rel=_REL.get(cat,0.3)
    if is_tool and cat not in ("outillage","soudure"): rel=min(rel,0.4)
    rel=round(rel,2); pid=str(r["product_handle"])
    _clusters.setdefault(dom,[]).append(pid)
    _prods.append({"id":pid,"name":r["product_title"],"inferred_function":_FUNC.get(cat,"composant"),
        "system_role":_role(cat,a),"system_position":a.get("system_position"),"technical_category":cat,"system_domain":dom,"relevance_score":rel,
        "iot_or_robotics_relevance": bool(rel>=0.5 and not (is_tool and cat in ("outillage","soudure","electrique"))),
        "compatibility_hints":_hints(a,is_tool),"similar_products_logic":_sim(cat,a),
        "complementary_products_logic":_comp(cat)})

_KB={"meta":{"total_products":len(_prods),"clusters":len(_clusters),
      "source":"inventory_export.csv (titres)","inference":"deterministe (moteur), aucune invention"},
 "clusters":_clusters,"products":_prods,
 "global_system_knowledge":{
  "iot_system_model":"CAPTEUR (input) -> CARTE/MCU (processing: Arduino/ESP32/Raspberry/STM32) -> COMM (LoRa/WiFi/BLE/433) -> supervision ; alimente par POWER, cable par CONNECTIQUE.",
  "robotics_system_model":"BATTERIE/ALIM -> CARTE -> DRIVER -> MOTEUR/SERVO (output) sur PLATEFORME (chassis/roues), CAPTEURS (ultrason/IMU) en boucle de retour.",
  "electronics_system_model":"COMPOSANTS (resistances/diodes/condensateurs/transistors) + MODULES (relais/afficheurs/convertisseurs) sur breadboard, en 3.3/5/12 V, pilotes par une carte.",
  "engineering_logic_discovered":"Systeme = INPUT(capteur)+PROCESSING(carte)+OUTPUT(moteur/led/relais)+ENERGY(alim/batterie)+SUPPORT(connectique/mecanique)+INTERFACE(afficheur/comm). Les OUTILS assemblent le systeme mais n'en font pas partie."},
 "rag_usage_guidelines":{
  "how_llm_should_use_this_data":"Verite figee : le LLM ne propose que des id presents ici, n'invente rien, ne modifie aucun champ ; il raisonne avec system_role/system_domain/compatibility_hints.",
  "retrieval_strategy":"Recuperer par system_domain + technical_category, filtrer par compatibility_hints (board/tension/connecteur), trier par relevance_score, garder le Top-K.",
  "ranking_strategy":"Similaires : meme technical_category + meme attribut definissant. Complementaires : domaines compatibles, compatibilite dure prioritaire, jamais un outil.",
  "anti_hallucination_rules":"(1) Sortie limitee aux id du dataset. (2) Pas de connaissance externe. (3) Si incertain -> 'unknown'. (4) Le score final vient du moteur deterministe, pas du LLM."}}

with open("rag_knowledge_base.json","w",encoding="utf-8") as _f:
    json.dump(_KB,_f,ensure_ascii=False,indent=1)
print(f"OK base RAG : {len(_prods)} produits, {len(_clusters)} clusters -> rag_knowledge_base.json")
print("Pertinents IoT/robotique :", sum(1 for p in _prods if p["iot_or_robotics_relevance"]), "/", len(_prods))
for _k,_v in sorted(_clusters.items(), key=lambda x:-len(x[1]))[:6]: print(f"   {_k:<26} {len(_v)}")


## Etape 12 - Modele de regles (hard rules)
Export `recommendation_rules.json` : tension / protocole / architecture + matrice de compatibilite + chaines systeme + `system_chain_edges`.

In [ ]:
# ============================================================================
# Etape 10 -- MODELE DE REGLES (recommendation_rules.json) : regles DURES, matrice de
#   compatibilite, definitions similarite/complementarite, chaines systeme. Machine-enforceable.
# ============================================================================
import json

_DEF = SmartRecoEngine.DEFINING
_matrix = {}
for _c in sorted(set(list(COMPLEMENTARITY_MAP.keys()) + list(_DEF.keys()))):
    _matrix[_c] = {
        "defining_attribute": _DEF.get(_c, None),
        "system_position": ("varie (selon module_fn)" if _c=="electronique" else _CAT_ROLE.get(_c, "autre")),
        "similar_with": [_c],                              # MEME categorie + meme attribut definissant
        "complementary_with": COMPLEMENTARITY_MAP.get(_c, []),
    }

RULES = {
 "version": "v5-system-roles",
 "hard_rules": [
   "TENSION : domaines logic(3.3/5V) / lv(12-48V) / mains(110-380V). Un appareil IoT (logic/lv) ne se complete JAMAIS avec du secteur (mains) -> rejet.",
   "ARCHITECTURE MCU : ESP32 / Arduino(AVR) / STM32 / Raspberry NE sont PAS substituables -> un similaire doit avoir le MEME board.",
   "OUTILS & INDUSTRIEL exclus des recommandations (marques connues + regex outil + blacklist 220V/automate/disjoncteur/FPGA).",
   "DISTANCE <= 2 sauts dans le graphe systeme : capteur->carte->afficheur OK ; capteur->moteur direct INTERDIT.",
   "MATCHING par MOT ENTIER (jamais sous-chaine : 'panne' != 'panneau', 'mpu' != 'impulsionnelle').",
   "LLM : ne retient que confidence >= 70 ; lexique strict (compatible / non compatible / necessite adaptation) ; AUCUNE invention.",
   "Le SCORE final vient du MOTEUR deterministe, jamais du LLM.",
 ],
 "compatibility_rules": [
   "board == board -> compatible (meme MCU).",
   "connector == connector -> compatible (meme connecteur).",
   "voltage_domain identique -> compatible.",
   "voltage_domain different (logic vs lv) -> necessite adaptation (convertisseur DC-DC).",
   "voltage_domain = mains face a logic/lv -> NON compatible (rejet dur).",
   "protocole sans fil (rftech : 433/lora/wifi/ble/nrf24/nfc) doit coincider pour les modules de communication.",
 ],
 "compatibility_matrix": _matrix,
 "similarity_rules": [
   "Meme technical_category ET meme attribut DEFINISSANT. Attributs definissants : " + json.dumps(_DEF, ensure_ascii=False) + ".",
   "Pas de cross-type pour carte/batterie/rf/capteur/led/mobilite (Arduino != ESP32, 18650 != pile bouton, gaz != debit).",
   "Niveaux techno separes : temperature NUMERIQUE (DHT/DS18B20) != ANALOGIQUE/INDUSTRIEL (PT100/thermocouple) ; ampoule != projecteur.",
   "Categories sans attribut definissant (connectique/mecanique/mobilite) : exiger un LIEN DE FAMILLE (token-modele ou mot significatif partage), sinon VIDE.",
   "VIDE > FAUX : si aucun substitut reel en stock, ne proposer AUCUN similaire.",
 ],
 "complementary_rules": [
   "Doit appartenir a la CHAINE SYSTEME (voir compatibility_matrix.complementary_with), a 2 sauts max.",
   "FILTRE PAR ROLE (system_chain_edges) : le role du complement doit etre un maillon ADJACENT valide du role source (ex: interface 'ruban LED' -> capteur 'thermostat' = REFUSE).",
   "Exige un signal de compatibilite REEL (board / connector / form_factor / chemistry / mot-cle / categorie-compagnon). La tension SEULE ne suffit pas.",
   "JAMAIS un outil ni du bruit industriel / domotique batiment.",
   "Diversite : pas de quasi-doublons dans la liste.",
 ],
 "system_role_map": _CAT_ROLE,
 "system_chain_edges": {_r: sorted(_e) for _r, _e in SmartRecoEngine.CHAIN_EDGES.items()},
 "system_chain_edges_note": ("ROLE source -> roles de complement ADMIS (maillon adjacent valide). Supports "
   "universels admis partout : " + ", ".join(sorted(SmartRecoEngine.SUPPORT_ROLES)) + ". 'electronique' affine "
   "son role via module_fn (display->interface ; relais/driver->actionneur ; convertisseur->alimentation ; "
   "capteur present->capteur ; sinon traitement)."),
 "system_chain_models": {
   "iot": "CAPTEUR (input, 3.3-5V) -> CARTE/MCU (processing) -> MODULE COMM (LoRa/WiFi/BLE/433) -> AFFICHEUR (interface) ; alimente par POWER (alim/batterie), cable par CONNECTIQUE.",
   "robotics": "BATTERIE/ALIM -> CARTE (controle) -> DRIVER -> MOTEUR/SERVO (output) -> STRUCTURE mecanique ; CAPTEURS (ultrason/IMU) en boucle de retour.",
   "energy": "BATTERIE -> BMS (protection) -> CHARGEUR -> REGULATION (buck/boost/alim) -> CHARGE.",
 },
 "recommendation_improvements": ("Regles dures de tension/protocole/architecture + limite a 2 sauts dans la chaine "
   "systeme + lexique de compatibilite deterministe (compatible/necessite adaptation) cote MOTEUR. Le LLM ne fait "
   "qu'expliquer dans ce cadre (confidence>=70, aucune invention). Resultat : strictement deterministe, zero "
   "hallucination, exact au plan ingenierie. Mesure moteur-seul stable a ~99% / 0 faute."),
}

with open("recommendation_rules.json","w",encoding="utf-8") as _f:
    json.dump(RULES,_f,ensure_ascii=False,indent=1)
print("OK modele de regles -> recommendation_rules.json")
print("  hard_rules:", len(RULES["hard_rules"]), "| matrice:", len(_matrix), "categories")
print("  chaines:", list(RULES["system_chain_models"].keys()))


## Bonus - Familles auto-decouvertes (clustering KMeans non supervise)
Preuve visuelle que le catalogue se structure naturellement en familles techniques.

In [ ]:
# BONUS -- familles auto-decouvertes (clustering non supervise, preuve de self-discovery)
from sklearn.cluster import KMeans
K_FAMILIES = 20
km = KMeans(n_clusters=K_FAMILIES, random_state=42, n_init=4).fit(engine.X)
engine.products["auto_family"] = km.labels_
print(f"{K_FAMILIES} familles decouvertes automatiquement par le modele (sans regles ecrites a la main).")

# Annexe — Déployer un LLM **privé, gratuit et illimité** pour la société (Ollama)

**Principe :** modèle à poids ouverts + serveur local = **0 € par token, volume illimité, 100 % privé** (aucune donnée du catalogue ne quitte l'entreprise — RGPD « by design »). Le serveur expose une **API compatible OpenAI**, donc le notebook ne change pas : il suffit de définir `LLM_BASE_URL` et `LLM_MODEL`.

## 1. Recommandation
- **Prototype / petit serveur → Ollama + `qwen2.5:7b`** : installation en 1 commande, CPU ou GPU, licence **Apache-2.0** (commercial libre). Largement suffisant ici (prompt court + 12 candidats → JSON).
- **Production PME → vLLM (Docker) + Qwen2.5-7B** quand vous dépassez ~5-6 requêtes simultanées (batching, sortie JSON garantie via `guided_json`). Même API : on change juste l'URL.

## 2. Modèles (sûrs pour usage commercial)
| Modèle | Taille | Français | Commercial | VRAM (q4) |
|---|---|---|---|---|
| **Qwen2.5-7B-Instruct** ⭐ | 7B | bon | **OUI** (Apache-2.0) | ~5-6 Go |
| Mistral 7B Instruct | 7B | bon | **OUI** (Apache-2.0) | ~5-6 Go |
| Qwen2.5-14B | 14B | très bon | **OUI** (Apache-2.0) | ~9-10 Go |
| Mistral Small 3 | 24B | excellent | **OUI** (Apache-2.0) | ~16-18 Go |
| Gemma 2/3 | 9-27B | bon | ⚠️ **non conseillé** (licence restrictive) | ~8-18 Go |
| Llama 3.1/3.3 | 8-70B | moyen/bon | ⚠️ **à éviter** (licence communautaire) | ~6-40 Go |

➡️ **Par défaut : Qwen2.5-7B** (français + JSON + Apache-2.0, zéro flou juridique).

## 3. Installation + branchement du notebook
**Installer Ollama** — Windows : installeur sur https://ollama.com/download · Linux : `curl -fsSL https://ollama.com/install.sh | sh`
```bash
ollama pull qwen2.5:7b          # ~5 Go (quantifié Q4)
curl http://localhost:11434/v1/models   # verifier (serveur auto-lance sur le port 11434)
```
**Pointer le notebook dessus** — mettre ces variables dans les *Secrets* Colab (ou `.env`, ou variables système du serveur) :
```
LLM_BASE_URL = http://localhost:11434/v1      # ou http://<serveur-interne>:11434/v1
LLM_MODEL    = qwen2.5:7b
```
Le notebook détecte automatiquement le serveur local et l'utilise **en priorité** (`[cerveau: llm:local]`). Aucune autre modification.

> **Migration vers vLLM** (sous charge) : rien à changer dans le code, on remplace juste l'URL/le modèle.
> `docker run --gpus all -p 8000:8000 vllm/vllm-openai:latest --model Qwen/Qwen2.5-7B-Instruct-AWQ`

## 4. Matériel + coût
| Config | Modèles | Débit | Coût | Pour qui |
|---|---|---|---|---|
| CPU seul (16-32 Go RAM) | 7B q4 | ~7-20 tok/s | 0 € | Test |
| **RTX 4060 Ti 16 Go** ⭐ | 7B **et** 14B q4 | 30-50 tok/s | ~500 $ | **PME (idéal)** |
| RTX 4090 24 Go | jusqu'à 24-32B | 30-80 tok/s | ~2500 $ | Qualité max |

➡️ Une machine + **RTX 4060 Ti 16 Go** couvre largement le besoin (réponse < 1 s sur prompts courts).

## 5. Sécurité / confidentialité
1. **Tout reste en interne** — aucun token ne quitte le réseau (RGPD by design).
2. **Exposition verrouillée** — API derrière reverse-proxy HTTPS, port restreint au VPN/sous-réseau interne, jamais ouvert sur Internet.
3. **Accès tracé** — clés/SSO + MFA pour les appelants internes, journalisation des requêtes.

**Résumé :** Ollama + Qwen2.5-7B (Apache-2.0) sur une RTX 4060 Ti 16 Go = gratuit, illimité, privé, API OpenAI-compatible. Le notebook s'y branche en 2 variables.


# 🎯 DEMONSTRATION FINALE - reponses du modele (`show_smart`)
Lance ces cellules **en dernier** pour voir les recommandations. Avec un LLM configure : `[cerveau IoT: llm:...]` + justifications ; sinon **moteur seul** (deterministe).

### Demo A - 6 produits varies
Le moteur (ou le LLM s'il est configure) propose similaires + complementaires pour 6 produits types.

In [ ]:
# == Demo : 6 produits varies. Si un LLM est configure, il choisit + EXPLIQUE (ligne ->).
# (HF gratuit limite les rafales ; pour enchainer beaucoup de produits, prends une cle Groq/Gemini gratuite.)
for _q in ["Arduino UNO", "ESP32", "Ruban LED RGB 5050", "Batterie 18650", "Capteur DHT11", "Module relais 4 canaux"]:
    show_smart(_q)


### Demo B - Teste TON produit
Edite la requete ci-dessous (titre, handle ou SKU) puis re-execute pour voir la reponse du modele.

In [ ]:
# Pour regler : edite regles_recommandation.csv puis re-execute a partir de l'Etape 5.
# Exemple de modif en direct :
COMPLEMENTARITY_MAP["mobilite"] = ["batterie","chargeur","moteur","carte"]
engine.comp_map = COMPLEMENTARITY_MAP
print("Carte de complementarite mise a jour")
show_smart("Drone")